# 02 — Building the `municipio_mes` table

This notebook takes the raw data collected by `01` (in `data/raw/<date>/`) and turns
it into the single table that feeds the model: **one row per municipality-month**,
from 2015 up to the freshness cutoff, with every variable and the outbreak labels.

The artifact is named `municipio_mes` — the table's *grain* (one row per
municipality per month). In data-science terms this is the *analytical base table*
(ABT); in the paper we call it the "consolidated municipality-month table". We name
it by the grain because that's the clearest: it says what each row represents.

**Design rules (same as `01`):**
- Markdown tells the story; code does one clear step at a time; genuinely complex
  logic (vaccination stitching, outbreak labels) becomes a small, well-named
  function right above the cell, explained in the markdown.
- Every step ends by showing a visible sample, for eyeball inspection.
- No magic numbers hidden away: every decision (freshness cutoff, outbreak
  definitions, minimum case count) is explicit and explained.

**Section order:** grid → per-source ingestion → join → missing vs zero →
outbreak labels → verification → output.

> Note on language: prose and comments are in English. Data column names are kept
> exactly as the source delivers them (`CLASSI_FIN`, `CODMUNRES`, `MUN_LPI`, ...) —
> those *are* the data, not our writing, so we never translate them.

## Section 0 — Setup: paths and freshness cutoff

Defines the paths, locates the most recent collection folder in `data/raw/<date>/`,
and **recomputes the freshness cutoff** from the collection date.

Why recompute here (instead of reading a manifest): we decided `02` is
self-contained. It redoes the 30/60-day math — notified cases consolidate in ~30
days, confirmed in ~60 — and sets the last trustworthy month of the base. Months
more recent than that cutoff are dropped by construction: they never become rows,
which prevents a still-incomplete month from being read as "no outbreak".

In [1]:
from pathlib import Path
from datetime import date, timedelta
import pandas as pd

# --- Paths ---
PROJECT_ROOT = Path.cwd().parent
RAW_BASE = PROJECT_ROOT / "data" / "raw"
LOOKUP_DIR = PROJECT_ROOT / "data" / "lookup"
OUT_DIR = PROJECT_ROOT / "data" / "master"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Find the most recent collection folder (data/raw/<date>/) ---
run_folders = sorted(p for p in RAW_BASE.iterdir() if p.is_dir())
if not run_folders:
    raise RuntimeError(f"No collection folders found in {RAW_BASE}")
RAW_DIR = run_folders[-1]
RUN_DATE = date.fromisoformat(RAW_DIR.name)   # the collection date, from the folder name
print(f"Using collection: {RAW_DIR.name}")
print(f"Sources present : {sorted(p.name for p in RAW_DIR.iterdir() if p.is_dir())}")

# --- Freshness cutoff (recomputed here; the 02 is self-contained) ---
# Notified cases consolidate in ~30 days, confirmed in ~60. We never trust a month
# whose end hasn't passed the cutoff, because partial data would read as "no outbreak".
CUTOFF_NOTIFIED_DAYS = 30
CUTOFF_CONFIRMED_DAYS = 60

def last_reliable_month(as_of: date, lag_days: int) -> tuple[int, int]:
    """Most recent (year, month) whose full month ended at least `lag_days` ago.
    Never returns the current month — partial data would degrade the labels."""
    # start from the first day of the current month, step back one month at a time
    y, m = as_of.year, as_of.month
    while True:
        # first day of month (y, m)
        first_of_month = date(y, m, 1)
        # last day of the PREVIOUS month = day before first_of_month
        last_day_prev = first_of_month - timedelta(days=1)
        if (as_of - last_day_prev).days >= lag_days:
            return (last_day_prev.year, last_day_prev.month)
        # step back a month
        m -= 1
        if m == 0:
            y, m = y - 1, 12

cutoff_notified = last_reliable_month(RUN_DATE, CUTOFF_NOTIFIED_DAYS)
cutoff_confirmed = last_reliable_month(RUN_DATE, CUTOFF_CONFIRMED_DAYS)

# The grid uses the CONFIRMED cutoff (conservative), since the paper uses confirmed cases.
GRID_START = (2015, 1)
GRID_END = cutoff_confirmed

print(f"\nCollection date        : {RUN_DATE}")
print(f"Cutoff (notified, 30d) : {cutoff_notified[0]}-{cutoff_notified[1]:02d}")
print(f"Cutoff (confirmed, 60d): {cutoff_confirmed[0]}-{cutoff_confirmed[1]:02d}  <- grid ends here")
print(f"Grid span              : {GRID_START[0]}-{GRID_START[1]:02d} to {GRID_END[0]}-{GRID_END[1]:02d}")

Using collection: 2026-09-09
Sources present : ['commuting', 'health', 'housing', 'ibge', 'mapbiomas', 'munic', 'nasa_power', 'primary_care_coverage', 'sih_sus', 'sinan', 'sinasc', 'sinisa', 'socioeconomic', 'territorial_areas', 'yellow_fever', 'yellow_fever_vaccination']

Collection date        : 2026-09-09
Cutoff (notified, 30d) : 2026-07
Cutoff (confirmed, 60d): 2026-06  <- grid ends here
Grid span              : 2015-01 to 2026-06


## Section 1 — The grid (the spine of the table)

Builds the cartesian product of **645 municipalities × every month** from 2015-01 up
to the freshness cutoff. Every row exists **before** any data is added.

Why: if we built the table from the notifications, a municipality-month with no
cases simply wouldn't exist — and "doesn't exist" is different from "zero cases".
The grid guarantees every municipality-month has a row; whatever is missing becomes
an explicit absence (NaN), handled in Section 4. This is the foundation; all sources
are joined on top of it.

In [2]:
# Step 1: the 645 SP municipalities, from the INMET lookup (the same 6->7 digit
# source the join will use). This is the authoritative municipality list.
lookup = pd.read_excel(LOOKUP_DIR / "municipios_sp_estacoes_inmet.xlsx")

In [3]:
# --- Section 1: the grid (the spine of the table) ---
# Every municipality-month exists as a row BEFORE any data is added.

# Step 2: unique municipalities. The lookup has one row per weather station, so a
# municipality can appear more than once — we take the distinct 7-digit codes.
municipios = (
    lookup[["Código Município Completo", "Nome_Município"]]
    .drop_duplicates(subset="Código Município Completo")
    .rename(columns={"Código Município Completo": "cod_ibge", "Nome_Município": "municipio"})
    .sort_values("cod_ibge")
    .reset_index(drop=True)
)
print(f"Municipalities: {len(municipios)}  (expected 645)")

# Step 3: the months, from GRID_START to GRID_END inclusive.
meses = pd.period_range(
    start=f"{GRID_START[0]}-{GRID_START[1]:02d}",
    end=f"{GRID_END[0]}-{GRID_END[1]:02d}",
    freq="M",
)
print(f"Months: {len(meses)}  ({meses[0]} to {meses[-1]})")

# Step 4: the cartesian product — every municipality × every month.
grid = municipios.merge(
    pd.DataFrame({"ano": meses.year, "mes": meses.month}),
    how="cross",
)
print(f"\nGrid shape: {grid.shape}  (expected {len(municipios) * len(meses):,} rows)")
print(f"Columns: {list(grid.columns)}")
grid.head(3)

Municipalities: 645  (expected 645)
Months: 138  (2015-01 to 2026-06)

Grid shape: (89010, 4)  (expected 89,010 rows)
Columns: ['cod_ibge', 'municipio', 'ano', 'mes']


,cod_ibge,municipio,ano,mes
0,3500105,Adamantina,2015,1
1,3500105,Adamantina,2015,2
2,3500105,Adamantina,2015,3


## Section 2 — Ingestion: each raw source into a clean table

Each of the 16 sources (the 15 + live births) is read from raw and reduced to the
key (municipality, year, month) — or (municipality, year) for the yearly ones. One
subsection per source, with a visible sample.

**Three NEW pieces of work here (they did not exist in the old pipeline):**
- **New parsers** for housing, IBGE and vaccination — `01` saved API JSON and
  consolidated CSVs, not the `.xlsx` files the old ingestion read. This is rewriting
  the reading, not porting.
- **Stitching the two vaccination halves** — TabNet (1994–2022) + Qlik panel
  (2023+), with a column marking which source each year came from. Work that never
  existed in code; we write it from scratch.
- **Live births → denominator** — aggregates SINASC by `CODMUNRES` (mother's
  municipality of residence), drops `350000` (unknown municipality).

**Locked decisions for this section:**
- SINAN: the arbovirus filter is by `CLASSI_FIN` (not CID) — dengue 10/11/12, chik
  13, zika 2. Temporal key = notification month (`DT_NOTIFIC`).
- Yellow fever: uses LPI (probable place of infection), and **this is where the
  empty-LPI count goes** — count how many SP cases have `MUN_LPI` filled vs. blank,
  to decide whether LPI costs many rows (the concern from the very start).
- **Maternal mortality: the column becomes `obitos_maternos` (honest name). With
  SINASC now collected, the real rate per 100k live births is computed** —
  numerator (deaths, through 2024) and denominator (births, through 2024) in the
  same window.
- 6→7 digit `join` via the INMET lookup; a municipality not in the dictionary is
  dropped (which also acts as the SP filter).

In [4]:
# --- Section 2: ingestion ---
# Each source is read from raw and reduced to the key (municipality, year, month),
# or (municipality, year) for yearly sources. One subsection per source.

# === Yellow fever ===
# Single national CSV, latin1, semicolon-separated. We saw it in 01: 2,912 cases
# since 1994, one row per confirmed case, keyed by probable place of infection (LPI).

yf_dir = RAW_DIR / "yellow_fever"
yf_file = next(yf_dir.glob("*.csv"))          # the fa_casoshumanos_1994-YYYY.csv we downloaded
print(f"Reading: {yf_file.name}")

yf_raw = pd.read_csv(yf_file, sep=";", encoding="latin1", low_memory=False)
print(f"Rows (national): {len(yf_raw):,}")
print(f"Columns: {list(yf_raw.columns)}")
yf_raw.head(3)

Reading: fa_casoshumanos_1994-2026.csv
Rows (national): 2,912
Columns: ['ID', 'MACRORREG_LPI', 'COD_UF_LPI', 'UF_LPI', 'COD_MUN_LPI', 'MUN_LPI', 'SEXO', 'IDADE', 'DT_IS', 'SE_IS', 'MES_IS', 'ANO_IS', 'MONITORAMENTO_IS', 'OBITO', 'DT_OBITO', 'Unnamed: 15']


,ID,MACRORREG_LPI,COD_UF_LPI,UF_LPI,COD_MUN_LPI,MUN_LPI,SEXO,IDADE,DT_IS,SE_IS,MES_IS,ANO_IS,MONITORAMENTO_IS,OBITO,DT_OBITO,Unnamed: 15
0,1.0,N,14.0,RR,140005.0,ALTO ALEGRE,M,NaN,29/11/1994,48.0,11.0,1994.0,1994/1995,SIM,01/12/1994,NaN
1,2.0,N,14.0,RR,140045.0,PACARAIMA,M,19,19/02/1995,8.0,2.0,1995.0,1994/1995,NÃO,NaN,NaN
2,3.0,NE,21.0,MA,210060.0,AMARANTE DO MARANHÃO,M,32,01/04/1995,13.0,4.0,1995.0,1994/1995,IGN,NaN,NaN


In [5]:
# --- Yellow fever: filter SP, count by municipality-month, check empty LPI ---

# The empty-LPI check (the concern from the very start): how many SP-attributable
# cases have a usable municipality code, and how many are blank? If many are blank,
# using LPI would cost us rows.
yf_sp = yf_raw[yf_raw["UF_LPI"] == "SP"].copy()
n_sp = len(yf_sp)
n_mun_missing = yf_sp["COD_MUN_LPI"].isna().sum()
print("=== Empty-LPI check (yellow fever, SP) ===")
print(f"SP cases (by LPI state)      : {n_sp}")
print(f"  with municipality filled   : {n_sp - n_mun_missing}")
print(f"  with municipality BLANK    : {n_mun_missing}  ({n_mun_missing / max(n_sp,1):.1%})")

# Build the clean table: cases and deaths per (municipality, year, month).
yf_sp = yf_sp.dropna(subset=["COD_MUN_LPI", "ANO_IS", "MES_IS"])
# COD_MUN_LPI is the 6-DIGIT DATASUS code, so the column is named cod6. Calling
# it cod_ibge would be a lie that joins silently to nothing: the grid is 7-digit.
yf_sp["cod6"] = yf_sp["COD_MUN_LPI"].astype(int).astype(str).str.zfill(6)
yf_sp["ano"] = yf_sp["ANO_IS"].astype(int)
yf_sp["mes"] = yf_sp["MES_IS"].astype(int)
yf_sp["is_obito"] = (yf_sp["OBITO"].astype(str).str.upper() == "SIM").astype(int)

yellow_fever = (
    yf_sp.groupby(["cod6", "ano", "mes"])
    .agg(fa_casos=("ID", "count"), fa_obitos=("is_obito", "sum"))
    .reset_index()
)
print(f"\nAggregated rows: {len(yellow_fever)}")
print(f"Year range: {yellow_fever['ano'].min()}–{yellow_fever['ano'].max()}")
print(f"Total SP cases aggregated: {yellow_fever['fa_casos'].sum()} (should match filled count above)")
yellow_fever.to_parquet(OUT_DIR / "yellow_fever_wide.parquet", index=False)
print(f"Saved: {OUT_DIR / 'yellow_fever_wide.parquet'}")

yellow_fever.head(5)

=== Empty-LPI check (yellow fever, SP) ===
SP cases (by LPI state)      : 766
  with municipality filled   : 765
  with municipality BLANK    : 1  (0.1%)

Aggregated rows: 255
Year range: 2000–2026
Total SP cases aggregated: 765 (should match filled count above)


,cod_ibge,ano,mes,fa_casos,fa_obitos
0,350030,2018,2,1,1
1,350040,2017,3,1,0
2,350040,2017,4,1,0
3,350050,2025,2,1,0
4,350170,2017,3,2,1


### Territorial areas (IBGE, 2025)

Land area in km² per municipality — **static**, one value per municipality, no time
dimension. Used as the denominator for incidence and population density.

- File comes national (5,575 rows); we keep the 645 SP municipalities.
- `CD_MUN` is IBGE's municipality code — we confirm below whether it's already the
  7-digit code (matching our grid) or the 6-digit one.
- `AR_MUN_2025` is the area in km².

In [6]:
# Read the territorial-areas file (IBGE, single .xls, national)
areas_dir = RAW_DIR / "territorial_areas"
area_file = next(areas_dir.iterdir())
areas_raw = pd.read_excel(area_file)

# Filter SP and reduce to (cod_ibge, area_km2).
areas_sp = areas_raw[areas_raw["NM_UF_SIGLA"] == "SP"].copy()
print(f"SP rows: {len(areas_sp)}  (expected 645)")

# Check the code width: is CD_MUN already 7 digits (like our grid) or 6?
sample_code = str(int(areas_sp["CD_MUN"].iloc[0]))
print(f"Sample CD_MUN: {sample_code}  ({len(sample_code)} digits)")

territorial_areas = (
    areas_sp[["CD_MUN", "AR_MUN_2025"]]
    .rename(columns={"CD_MUN": "cod_ibge", "AR_MUN_2025": "area_km2"})
    .astype({"cod_ibge": int})
    .reset_index(drop=True)
)
print(f"\nClean rows: {len(territorial_areas)}")
print(f"Area range: {territorial_areas['area_km2'].min():.1f} – {territorial_areas['area_km2'].max():.1f} km²")
territorial_areas.to_parquet(OUT_DIR / "territorial_areas_wide.parquet", index=False)
print(f"Saved: {OUT_DIR / 'territorial_areas_wide.parquet'}")

territorial_areas.head(3)

SP rows: 645  (expected 645)
Sample CD_MUN: 3500105  (7 digits)

Clean rows: 645
Area range: 3.4 – 1978.3 km²


,cod_ibge,area_km2
0,3500105,411.987
1,3500204,211.055
2,3500303,474.554


### SINAN — dengue, zika, chikungunya (the heart of the data)

The most important source: the outbreak labels come from here. Three diseases, one
`.dbc` file per disease per year (FINAIS = consolidated/immutable, PRELIM = revised).

Key decisions (locked earlier):
- **Arbovirus filter is by `CLASSI_FIN`, not CID**: dengue 10/11/12, chikungunya 13,
  zika 2 (5 = discarded).
- Temporal key = **notification month** (`DT_NOTIFIC`).
- Death = `EVOLUCAO` in {2,3,4}; hospitalization = `HOSPITALIZ == '1'` (but the zika
  `.dbc` may lack this field — handle nulls so "field absent" ≠ "never hospitalized").
- 6→7 digit code (`ID_MN_RESI`) is reconciled in the join (Section 3), not here.

First, we open ONE file to confirm the real column names before writing the filter.

In [7]:
import tempfile
import gc
import dbfread
from pyreaddbc import dbc2dbf

sinan_dir = RAW_DIR / "sinan"

def read_sinan_filtered(path, classi_ok, keep_cols):
    """Read a SINAN .dbc in STREAMING mode, keeping only SP + confirmed rows.
    Never materializes the whole national file in memory."""
    with tempfile.NamedTemporaryFile(suffix=".dbf", delete=False) as tmp:
        tmp_dbf = tmp.name
    dbc2dbf(str(path), tmp_dbf)

    rows = []
    # load=False makes DBF iterate record-by-record instead of loading everything
    table = dbfread.DBF(tmp_dbf, encoding="latin1", load=False)
    for rec in table:
        # filter as we read: SP only, confirmed only
        uf = str(rec.get("SG_UF", "")).strip()
        if uf != "35":
            continue
        try:
            classi = int(rec.get("CLASSI_FIN") or -1)
        except (ValueError, TypeError):
            continue
        if classi not in classi_ok:
            continue
        rows.append({c: rec.get(c) for c in keep_cols})

    Path(tmp_dbf).unlink(missing_ok=True)
    gc.collect()
    return pd.DataFrame(rows)

#### SINAN — Step A: read and stack the individual records

We read all 36 files (3 diseases × 12 years), apply the `CLASSI_FIN` filter per
disease, and stack the confirmed cases into ONE DataFrame of individual records —
**without aggregating yet**.

Why two steps: the same (municipality, month) key appears in more than one file
(e.g. `DENGBR25` carries some 2026 notifications). If we computed the latency
median file-by-file it would be wrong. So we stack everything first, then aggregate
in Step B — that way the latency is computed over all records of a key at once.

From each record we keep only what we need: disease, residence municipality
(`ID_MN_RESI`), notification date (`DT_NOTIFIC`, the temporal key), symptom-onset
date (`DT_SIN_PRI`, for latency), outcome (`EVOLUCAO`), hospitalization
(`HOSPITALIZ`). This reads 36 files including large dengue ones — **expect a few
minutes.**

In [8]:
import gc
import time

DISEASE_PREFIX = {"dengue": "DENGBR", "zika": "ZIKABR", "chikungunya": "CHIKBR"}
CLASSI_FIN_OK = {"dengue": {10, 11, 12}, "zika": {2}, "chikungunya": {13}}
KEEP = ["ID_MN_RESI", "DT_NOTIFIC", "DT_SIN_PRI", "EVOLUCAO", "HOSPITALIZ"]

# Checkpoint dir: each file's filtered result saved here, so a crash never loses progress.
sinan_ckpt = RAW_DIR.parent.parent / "master" / "sinan_checkpoints"
sinan_ckpt.mkdir(parents=True, exist_ok=True)

def years_on_disk(prefix):
    import re
    ys = []
    for f in sinan_dir.glob(f"{prefix}*.dbc"):
        m = re.search(rf"{prefix}(\d{{2}})\.dbc$", f.name, re.IGNORECASE)
        if m:
            ys.append(2000 + int(m.group(1)))
    return sorted(ys)

t0 = time.time()
for disease, prefix in DISEASE_PREFIX.items():
    for year in years_on_disk(prefix):
        ckpt = sinan_ckpt / f"{disease}_{year}.parquet"
        if ckpt.exists():
            print(f"  {disease:12} {year}: checkpoint exists, skipping  ({time.time()-t0:.0f}s)")
            continue
        f = sinan_dir / f"{prefix}{str(year)[-2:]}.dbc"
        df = read_sinan_filtered(f, CLASSI_FIN_OK[disease], KEEP)
        df.to_parquet(ckpt)                       # save this file's result immediately
        print(f"  {disease:12} {year}: {len(df):>9,} SP cases -> saved  ({time.time()-t0:.0f}s)")
        del df
        gc.collect()

print(f"\nAll files processed in {time.time()-t0:.0f}s")

  dengue       2015:   683,007 SP cases -> saved  (87s)
  dengue       2016:   177,272 SP cases -> saved  (164s)
  dengue       2017:     6,319 SP cases -> saved  (177s)
  dengue       2018:    14,745 SP cases -> saved  (191s)
  dengue       2019:   410,112 SP cases -> saved  (273s)
  dengue       2020:   195,342 SP cases -> saved  (326s)
  dengue       2021:   145,242 SP cases -> saved  (381s)
  dengue       2022:   331,612 SP cases -> saved  (457s)
  dengue       2023:   326,700 SP cases -> saved  (544s)
  dengue       2024: 2,153,761 SP cases -> saved  (891s)
  dengue       2025:   886,021 SP cases -> saved  (979s)
  dengue       2026:    58,758 SP cases -> saved  (1003s)
  zika         2015:       118 SP cases -> saved  (1004s)
  zika         2016:     5,058 SP cases -> saved  (1010s)
  zika         2017:     1,242 SP cases -> saved  (1010s)
  zika         2018:     1,137 SP cases -> saved  (1011s)
  zika         2019:     2,845 SP cases -> saved  (1011s)
  zika         2020:      

#### SINAN — Step B: aggregate to municipality-month

Read the 36 checkpoints, stack them (they're small now — already filtered to SP +
confirmed), and aggregate by (disease, municipality, notification-month) into:
- **casos** — number of confirmed cases
- **obitos** — EVOLUCAO in {2,3,4}
- **internacoes** — HOSPITALIZ == '1' (null-tolerant; zika files may lack the field)
- **latencia** — median and p90 of (DT_NOTIFIC − DT_SIN_PRI), valid days 0–365 only

Latency is computed here, over all records of a key at once, which is why we stacked
first (the old pipeline warned that per-file medians are wrong — the same key spans
multiple files). Output: one row per (disease, cod_ibge, ano, mes), pivoted so each
disease's columns are prefixed.

In [14]:
# --- SINAN Step B: read checkpoints and aggregate ---
frames = []
for ckpt in sorted(sinan_ckpt.glob("*.parquet")):
    disease = ckpt.stem.rsplit("_", 1)[0]        # "dengue_2024" -> "dengue"
    df = pd.read_parquet(ckpt)
    df["disease"] = disease
    frames.append(df)

sinan_all = pd.concat(frames, ignore_index=True)
print(f"Total stacked records: {len(sinan_all):,}")

# Parse fields
dt_notif = pd.to_datetime(sinan_all["DT_NOTIFIC"], errors="coerce")
dt_sin   = pd.to_datetime(sinan_all["DT_SIN_PRI"], errors="coerce")
sinan_all["cod6"] = sinan_all["ID_MN_RESI"].astype(str).str.strip()
sinan_all["ano"]  = dt_notif.dt.year
sinan_all["mes"]  = dt_notif.dt.month
sinan_all["is_obito"] = pd.to_numeric(sinan_all["EVOLUCAO"], errors="coerce").isin([2, 3, 4])
sinan_all["is_hosp"]  = sinan_all["HOSPITALIZ"].astype(str).str.strip() == "1"
lat = (dt_notif - dt_sin).dt.days
sinan_all["latencia"] = lat.where((lat >= 0) & (lat <= 365))

# drop rows with no valid notification month
sinan_all = sinan_all.dropna(subset=["ano", "mes"])
sinan_all["ano"] = sinan_all["ano"].astype(int)
sinan_all["mes"] = sinan_all["mes"].astype(int)

# aggregate by (disease, municipality, month) — vectorized
agg = sinan_all.groupby(["disease", "cod6", "ano", "mes"]).agg(
    casos=("is_obito", "size"),
    obitos=("is_obito", "sum"),
    internacoes=("is_hosp", "sum"),
    latencia_mediana=("latencia", "median"),
    latencia_p90=("latencia", lambda s: s.quantile(0.90)),
).reset_index()

print(f"Aggregated rows: {len(agg):,}")
print(f"By disease:\n{agg.groupby('disease')['casos'].sum()}")
agg.head(5)

Total stacked records: 5,446,495
Aggregated rows: 56,420
By disease:
disease
chikungunya      39106
dengue         5388891
zika             18498
Name: casos, dtype: int64


,disease,cod6,ano,mes,casos,obitos,internacoes,latencia_mediana,latencia_p90
0,chikungunya,350010,2016,3,1,0,0,7.0,7.0
1,chikungunya,350010,2016,9,1,0,1,17.0,17.0
2,chikungunya,350010,2017,8,1,0,0,4.0,4.0
3,chikungunya,350010,2018,10,3,0,0,10.0,10.0
4,chikungunya,350010,2022,8,1,0,0,6.0,6.0


In [15]:
# --- SINAN: pivot to one row per (municipality, month), disease columns prefixed ---
value_cols = ["casos", "obitos", "internacoes", "latencia_mediana", "latencia_p90"]

sinan_wide = agg.pivot_table(
    index=["cod6", "ano", "mes"],
    columns="disease",
    values=value_cols,
    observed=True,
)
# flatten the column MultiIndex: ("casos","dengue") -> "dengue_casos"
sinan_wide.columns = [f"{disease}_{metric}" for metric, disease in sinan_wide.columns]
sinan_wide = sinan_wide.reset_index()

print(f"SINAN wide: {sinan_wide.shape}")
print(f"Columns: {list(sinan_wide.columns)}")
sinan_wide.to_parquet(OUT_DIR / "sinan_wide.parquet", index=False)
print(f"Saved: {OUT_DIR / 'sinan_wide.parquet'}")

sinan_wide.head(3)

SINAN wide: (48831, 18)
Columns: ['cod6', 'ano', 'mes', 'chikungunya_casos', 'dengue_casos', 'zika_casos', 'chikungunya_internacoes', 'dengue_internacoes', 'zika_internacoes', 'chikungunya_latencia_mediana', 'dengue_latencia_mediana', 'zika_latencia_mediana', 'chikungunya_latencia_p90', 'dengue_latencia_p90', 'zika_latencia_p90', 'chikungunya_obitos', 'dengue_obitos', 'zika_obitos']


,cod6,ano,mes,chikungunya_casos,dengue_casos,zika_casos,chikungunya_internacoes,dengue_internacoes,zika_internacoes,chikungunya_latencia_mediana,dengue_latencia_mediana,zika_latencia_mediana,chikungunya_latencia_p90,dengue_latencia_p90,zika_latencia_p90,chikungunya_obitos,dengue_obitos,zika_obitos
0,,2015,2,NaN,1.0,NaN,NaN,0.0,NaN,NaN,6.0,NaN,NaN,6.0,NaN,NaN,0.0,NaN
1,,2015,3,NaN,1.0,NaN,NaN,0.0,NaN,NaN,5.0,NaN,NaN,5.0,NaN,NaN,0.0,NaN
2,,2015,4,NaN,1.0,NaN,NaN,0.0,NaN,NaN,7.0,NaN,NaN,7.0,NaN,NaN,0.0,NaN


### SIH-SUS — hospitalizations (the second heavy .dbc source)

Hospital admissions paid by SUS. Here it contributes **hospitalizations by
arbovirus**, per municipality-month. Like SINAN it's `.dbc`, so we reuse the
streaming reader. Differences from SINAN:
- **139 files**, one per month (`RDSP{YYMM}.dbc`, 2015-01 to 2026-07) — already SP
  only (the `SP` in the name), so no UF filter needed.
- Disease is filtered by **CID-10 in `DIAG_PRINC`** (not CLASSI_FIN): dengue
  A90/A91, chikungunya A92.0, zika A92.5/A92.8, yellow fever A95. The field comes
  **padded**, so we strip and match by prefix.
- Municipality is `MUNIC_RES` (6-digit), month comes from the file name.

First, open ONE file to confirm the real column names.

In [ ]:
# The SIH-SUS raw folder. Everything else this source needs — the CID-10
# mapping, the checkpoint dir and the streaming reader — is defined in the
# next cell, which is where the work happens.
sih_dir = RAW_DIR / "sih_sus"
print(f"SIH files: {len(list(sih_dir.glob('RDSP*.dbc')))} monthly .dbc")


In [17]:
import gc, time, re

# CID-10 -> disease mapping (DIAG_PRINC comes without the dot, padded)
def cid_to_disease(cid):
    cid = str(cid).strip().upper()
    if cid.startswith(("A90", "A91")): return "dengue"
    if cid.startswith("A920"):          return "chikungunya"
    if cid.startswith(("A925", "A928")): return "zika"
    if cid.startswith("A95"):           return "febre_amarela"
    return None

sih_ckpt = sinan_ckpt.parent / "sih_checkpoints"
sih_ckpt.mkdir(parents=True, exist_ok=True)

def read_sih_month(path):
    """Streaming read of one SIH .dbc, keeping only arbovirus hospitalizations.
    Returns rows with municipality, disease, and death flag."""
    import tempfile
    with tempfile.NamedTemporaryFile(suffix=".dbf", delete=False) as tmp:
        tmp_dbf = tmp.name
    dbc2dbf(str(path), tmp_dbf)
    rows = []
    for rec in dbfread.DBF(tmp_dbf, encoding="latin1", load=False):
        disease = cid_to_disease(rec.get("DIAG_PRINC", ""))
        if disease is None:
            continue
        rows.append({
            "cod6": str(rec.get("MUNIC_RES", "")).strip(),
            "disease": disease,
            "morte": 1 if str(rec.get("MORTE", "")).strip() == "1" else 0,
        })
    Path(tmp_dbf).unlink(missing_ok=True)
    gc.collect()
    return pd.DataFrame(rows)

t0 = time.time()
for f in sorted(sih_dir.glob("RDSP*.dbc")):
    m = re.search(r"RDSP(\d{2})(\d{2})\.dbc$", f.name, re.IGNORECASE)
    yy, mm = int(m.group(1)), int(m.group(2))
    ano, mes = 2000 + yy, mm
    ckpt = sih_ckpt / f"sih_{ano}_{mes:02d}.parquet"
    if ckpt.exists():
        continue
    df = read_sih_month(f)
    df["ano"] = ano
    df["mes"] = mes
    df.to_parquet(ckpt)
    if len(df):
        print(f"  {ano}-{mes:02d}: {len(df):>5,} arbovirus hosp -> saved  ({time.time()-t0:.0f}s)")
    del df
    gc.collect()

print(f"\nAll 139 SIH files processed in {time.time()-t0:.0f}s")

  2015-01:   508 arbovirus hosp -> saved  (12s)
  2015-02: 1,350 arbovirus hosp -> saved  (24s)
  2015-03: 2,872 arbovirus hosp -> saved  (36s)
  2015-04: 3,757 arbovirus hosp -> saved  (49s)
  2015-05: 3,200 arbovirus hosp -> saved  (62s)
  2015-06: 1,500 arbovirus hosp -> saved  (74s)
  2015-07:   574 arbovirus hosp -> saved  (87s)
  2015-08:   213 arbovirus hosp -> saved  (99s)
  2015-09:   125 arbovirus hosp -> saved  (111s)
  2015-10:   129 arbovirus hosp -> saved  (123s)
  2015-11:   140 arbovirus hosp -> saved  (134s)
  2015-12:   285 arbovirus hosp -> saved  (146s)
  2016-01:   543 arbovirus hosp -> saved  (157s)
  2016-02:   957 arbovirus hosp -> saved  (168s)
  2016-03: 1,042 arbovirus hosp -> saved  (180s)
  2016-04: 1,059 arbovirus hosp -> saved  (193s)
  2016-05:   613 arbovirus hosp -> saved  (205s)
  2016-06:   212 arbovirus hosp -> saved  (217s)
  2016-07:    94 arbovirus hosp -> saved  (230s)
  2016-08:    39 arbovirus hosp -> saved  (242s)
  2016-09:    39 arbovirus h

In [18]:
# --- SIH Step B: aggregate and pivot ---
sih_frames = [pd.read_parquet(p) for p in sorted(sih_ckpt.glob("*.parquet"))]
sih_all = pd.concat(sih_frames, ignore_index=True)
print(f"Total hospitalizations: {len(sih_all):,}")

# aggregate: count of hospitalizations per (disease, municipality, month)
sih_agg = sih_all.groupby(["disease", "cod6", "ano", "mes"]).size().reset_index(name="internacoes_sih")
print(f"By disease:\n{sih_all.groupby('disease').size()}")

# pivot: one row per (municipality, month), one column per disease
sih_wide = sih_agg.pivot_table(
    index=["cod6", "ano", "mes"], columns="disease", values="internacoes_sih", observed=True
)
sih_wide.columns = [f"{d}_internacoes_sih" for d in sih_wide.columns]
sih_wide = sih_wide.reset_index()

# save it, like we did with SINAN
sih_wide.to_parquet(sinan_ckpt.parent / "sih_wide.parquet")
print(f"\nSIH wide: {sih_wide.shape}")
print(f"Columns: {list(sih_wide.columns)}")
sih_wide.head(3)

Total hospitalizations: 111,448
By disease:
disease
chikungunya         179
dengue           110765
febre_amarela       367
zika                137
dtype: int64

SIH wide: (19409, 7)
Columns: ['cod6', 'ano', 'mes', 'chikungunya_internacoes_sih', 'dengue_internacoes_sih', 'febre_amarela_internacoes_sih', 'zika_internacoes_sih']


,cod6,ano,mes,chikungunya_internacoes_sih,dengue_internacoes_sih,febre_amarela_internacoes_sih,zika_internacoes_sih
0,110013,2025,1,NaN,1.0,NaN,NaN
1,110020,2024,6,NaN,1.0,NaN,NaN
2,110020,2025,9,NaN,1.0,NaN,NaN


### Population density (derived: IBGE areas + IBGE population)

People per km². This is one of the strongest structural predictors we have:
*Aedes aegypti* flies only a few hundred metres in its lifetime, so how tightly
people are packed decides how far a single infected mosquito can carry an
outbreak.

**This source is derived, not collected.** There is no `densidade` folder in raw —
it is the quotient of two sources `01` did collect: territorial area (read above)
and IBGE resident population. The cell below re-reads both from raw rather than
reusing variables already in memory, so it survives a kernel restart on its own.

Things that matter here:

- **We compute density per YEAR, where the old pipeline froze it as a single
  static value.** The old `ingestion/densidade.py` took the most recent population
  year and pinned that one number to every row, back to 2015. Population is
  published annually and moves, so we key this table on `(cod_ibge, ano)` and let
  each year carry its own density. Flagging the change explicitly because it makes
  this variable behave differently from the old base.
- **There is no 2023.** IBGE published no population estimate for that year — the
  series jumps 2022 (census *count*) to 2024 (estimate). So density has no 2023
  either. We leave the hole exactly as it is; deciding whether to carry a value
  forward is Section 4's call, not this cell's.
- **Area comes from the 2025 edition and is held constant across years.**
  Municipal boundaries only move with an emancipation, merger or boundary
  revision, none of which happened in São Paulo inside this window.
- **The area column name carries the year** (`AR_MUN_2025`), so we find it by
  prefix instead of hard-coding it — the same trap `01` documented for the sheet
  and file names.
- **Both codes are already 7 digits** (`CD_MUN` and `localidade_id`), so nothing
  here needs the 6→7 conversion. That conversion belongs to Section 3 anyway, and
  applies to the DATASUS sources.

In [ ]:
# --- Population density: area (IBGE) + resident population (IBGE) ---
# Read both from raw so this cell stands on its own after a kernel restart.

# Step 1: territorial area, from the national .xls. The area column carries the
# edition year in its name (AR_MUN_2025), so match it by prefix.
areas_file = next((RAW_DIR / "territorial_areas").iterdir())
areas_all = pd.read_excel(areas_file)
area_col = next(c for c in areas_all.columns if str(c).upper().startswith("AR_MUN"))
print(f"Area file  : {areas_file.name}   area column: {area_col!r}")

areas = (
    areas_all[areas_all["NM_UF_SIGLA"] == "SP"][["CD_MUN", area_col]]
    .rename(columns={"CD_MUN": "cod_ibge", area_col: "area_km2"})
    .dropna(subset=["cod_ibge"])
    .astype({"cod_ibge": "int64"})
    .reset_index(drop=True)
)
print(f"SP municipalities with an area: {len(areas)}  (expected 645)")

# Step 2: resident population per year, from the consolidated CSV 01 wrote.
# `fonte` records whether a year is a census count or an estimate — we keep it in
# the printout below but not in the output table, since it belongs to the IBGE
# section, which owns the population variable itself.
pop = pd.read_csv(RAW_DIR / "ibge" / "sidra_population.csv")
pop = (
    pop[["localidade_id", "ano", "pop_residente", "fonte"]]
    .rename(columns={"localidade_id": "cod_ibge"})
    .astype({"cod_ibge": "int64", "ano": "int64"})
)
years_present = sorted(pop["ano"].unique())
print(f"Population years present: {years_present}")
missing_years = [y for y in range(min(years_present), max(years_present) + 1)
                 if y not in set(years_present)]
print(f"Years IBGE never published: {missing_years}  -> density will have no value there")

# Step 3: the quotient. Inner join on purpose: a municipality-year without a
# population has no density, and inventing one would be worse than leaving it out.
densidade = pop.merge(areas, on="cod_ibge", how="inner", validate="m:1")
densidade["densidade_hab_km2"] = (densidade["pop_residente"] / densidade["area_km2"]).round(2)
densidade = (
    densidade[["cod_ibge", "ano", "densidade_hab_km2"]]
    .sort_values(["cod_ibge", "ano"])
    .reset_index(drop=True)
)

# Step 4: look at what came out.
print(f"\nDensity table: {densidade.shape}")
print(f"Columns: {list(densidade.columns)}")
print(f"Municipalities: {densidade['cod_ibge'].nunique()}  |  years: "
      f"{densidade['ano'].min()}-{densidade['ano'].max()} "
      f"({densidade['ano'].nunique()} of them)")
print(f"Density range: {densidade['densidade_hab_km2'].min():.1f} - "
      f"{densidade['densidade_hab_km2'].max():,.1f} hab/km²")

# Sanity check worth doing every time: the extremes should be recognisable.
extremes = densidade[densidade["ano"] == densidade["ano"].max()].nlargest(
    3, "densidade_hab_km2")
print(f"\nDensest municipalities in {densidade['ano'].max()}:")
print(extremes.to_string(index=False))

densidade.to_parquet(OUT_DIR / "densidade_wide.parquet", index=False)
print(f"\nSaved: {OUT_DIR / 'densidade_wide.parquet'}")

densidade.head(3)

### Housing: favelas and subnormal settlements (IBGE census, 2010 + 2022)

How many favelas a municipality has and how many people live in them. It belongs
in this model because precarious, dense housing with unreliable piped water is
exactly where *Aedes aegypti* breeds: water stored in drums and tanks because the
tap cannot be trusted is the mosquito's preferred nursery.

`01` already flattened the four SIDRA API answers into `sidra_housing.csv`
(the four raw JSONs stay beside it as provenance), so reading here is trivial —
note this is a different shape from the old pipeline, which read four `.xlsx`
files downloaded by hand.

Things that matter here:

- **This is a snapshot source, not an annual series.** Only two years exist,
  2010 and 2022, because those are the censuses. The key is `(cod_ibge, ano)` like
  any yearly source, but Section 3 cannot merge it as one: it has to carry each
  snapshot forward until the next one, and back-fill 2015–2021 from 2010. Joining
  on year alone would leave every other year empty.
- **The concept itself changed between the censuses.** IBGE renamed "aglomerado
  subnormal" (2010) to "favela e comunidade urbana" (2022) and redrew the
  delimitation criteria. Municipalities with any record go from **60 to 92**. Part
  of that jump is real growth and part is the new definition, and nothing
  separates them. The cell prints both counts so the break stays visible.
- **A municipality with no row means NO RECORD, not zero.** In 2022, 553 of the
  645 have no row at all. Writing zeros there would assert "there is no favela in
  this municipality", when the census merely did not identify one. We keep the
  absence; whether it becomes a zero is Section 4's decision, not this cell's, and
  it is the single most consequential thing about this source.
- **`localidade_id` is already the 7-digit code**, so nothing needs converting.

In [ ]:
# --- Housing: favelas / aglomerados subnormais (two census snapshots) ---
# 01 flattened the four SIDRA answers into one CSV; the raw JSONs sit next to it.

housing_file = RAW_DIR / "housing" / "sidra_housing.csv"
housing_raw = pd.read_csv(housing_file)
print(f"Reading: {housing_file.name}")
print(f"Rows: {len(housing_raw)}   columns: {list(housing_raw.columns)}")

habitacao = (
    housing_raw[["localidade_id", "ano", "num_favelas", "pop_favelas"]]
    .rename(columns={"localidade_id": "cod_ibge"})
    .astype({"cod_ibge": "int64", "ano": "int64"})
    .sort_values(["cod_ibge", "ano"])
    .reset_index(drop=True)
)

# Step 2: the concept change, made visible. 60 -> 92 municipalities is the number
# to remember: it is growth AND a redrawn definition, inseparably.
print("\n=== Municipalities with a record, by census ===")
by_census = habitacao.groupby("ano").agg(
    municipios=("cod_ibge", "nunique"),
    favelas=("num_favelas", "sum"),
    populacao=("pop_favelas", "sum"),
)
print(by_census.to_string())

# Step 3: state the absence out loud rather than letting it look like zero.
for ano in sorted(habitacao["ano"].unique()):
    com_registro = habitacao.loc[habitacao["ano"] == ano, "cod_ibge"].nunique()
    print(f"{ano}: {com_registro} of 645 municipalities have a record; "
          f"the other {645 - com_registro} have NO ROW "
          f"(no record — not a zero)")

# Step 4: look at what came out.
print(f"\nHousing table: {habitacao.shape}")
print(f"Columns: {list(habitacao.columns)}")
print(f"Census years: {sorted(habitacao['ano'].unique())}")

habitacao.to_parquet(OUT_DIR / "habitacao_wide.parquet", index=False)
print(f"Saved: {OUT_DIR / 'habitacao_wide.parquet'}")

habitacao.head(3)

### Population and GDP (IBGE, SIDRA)

Two annual municipal series. Population is not just another feature: it is **the
denominator of incidence**, so it flows straight into how outbreak labels are
defined in Section 5. A wrong population silently rewrites the target variable.

`01` consolidated the three SIDRA answers into two CSVs (the raw JSONs stay as
provenance), so this reads clean — again a different shape from the old pipeline,
which read `.xlsx` files pulled by hand.

Things that matter here:

- **The population series had been frozen at 2021 for five years in the old base,
  and nobody noticed.** The estimates table (6579) has **no 2022 and no 2023 at
  all**; the old code forward-filled 2021 into both. `01` stitched the 2022 census
  *count* (table 4709) into the series, which leaves exactly one real hole: **2023**.
  The `fonte` column records which years are a census count and which an estimate.
- **GDP genuinely lags** — it ends in 2023 — so `pib_per_capita` is null from 2024
  on. There the *numerator* is missing, not the denominator: a different problem
  from the population gap, and worth not confusing.
- **We do not fill either gap here.** Carrying values forward is a join-time
  decision (Section 4), and doing it in the ingestion would hide which years were
  actually published.

In [ ]:
# --- IBGE: resident population and municipal GDP ---
pop = pd.read_csv(RAW_DIR / "ibge" / "sidra_population.csv")
gdp = pd.read_csv(RAW_DIR / "ibge" / "sidra_gdp.csv")
print(f"Population rows: {len(pop):,}   GDP rows: {len(gdp):,}")

# "fonte" says whether a year is a census count or an estimate. Renamed because
# the vaccination source ships a column called "fonte" too, and the join would
# silently turn the two into fonte_x / fonte_y.
pop = (pop[["localidade_id", "ano", "pop_residente", "fonte"]]
       .rename(columns={"localidade_id": "cod_ibge", "fonte": "pop_fonte"}))
gdp = (gdp[["localidade_id", "ano", "pib_mil_reais"]]
       .rename(columns={"localidade_id": "cod_ibge"}))

# Outer join: the two series cover different year ranges on purpose, and we want
# to see both edges rather than silently intersect them away.
ibge = pop.merge(gdp, on=["cod_ibge", "ano"], how="outer", validate="1:1")
ibge = ibge.astype({"cod_ibge": "int64", "ano": "int64"})

# GDP is in thousands of reais; per-capita is in reais.
ibge["pib_per_capita"] = (ibge["pib_mil_reais"] * 1000 / ibge["pop_residente"]).round(2)

ibge = (ibge[["cod_ibge", "ano", "pop_residente", "pop_fonte", "pib_mil_reais",
              "pib_per_capita"]]
        .sort_values(["cod_ibge", "ano"]).reset_index(drop=True))

# Step 2: show what each year actually has. This table is the whole point — it is
# where a frozen population would have been visible five years earlier.
print("\n=== Coverage by year (645 = complete) ===")
coverage = ibge.groupby("ano").agg(
    pop_municipios=("pop_residente", "count"),
    pop_fonte=("pop_fonte", lambda s: "/".join(sorted(set(s.dropna())))),
    pib_municipios=("pib_mil_reais", "count"),
)
print(coverage.to_string())

anos = sorted(ibge["ano"].unique())
sem_pop = [a for a in range(min(anos), max(anos) + 1)
           if ibge.loc[ibge["ano"] == a, "pop_residente"].notna().sum() == 0]
print(f"\nYears with NO published population: {sem_pop}  (left empty on purpose)")

print(f"\nIBGE table: {ibge.shape}   columns: {list(ibge.columns)}")
ibge.to_parquet(OUT_DIR / "ibge_wide.parquet", index=False)
print(f"Saved: {OUT_DIR / 'ibge_wide.parquet'}")

ibge.head(3)

### Socioeconomic: CAPAG, IDH-M and Gini

What a municipality can actually afford to do about an outbreak.

- **CAPAG** (National Treasury) — a yearly A/B/C/D rating of municipal fiscal
  health. A municipality rated C does not run a vector-control programme the way
  an A does. Collected by `01`, one spreadsheet per year.
- **IDH-M** and **Gini** — census 2010, static per municipality. Neither is
  collectable: the Atlas site is down and the surviving copies live behind
  BigQuery credentials, so both are **manual assets** in `data/lookup/`.

**This source is expected to be PARTIAL, and that is not a failure.** The cell
reports what is missing instead of raising.

Things that matter here:

- **The CAPAG spreadsheets have at least three different layouts.** The header sits
  on row 0 in some files and row 2 in others; the code column is `Cod.IBGE` in
  some and `Código Município Completo` in others; the grade column moves too. So
  we **find** the header row and the columns by keyword rather than assuming, the
  same principle used for the SINISA sheets in `01`.
- **The reference year comes from the file name — except for one file.**
  `capag-municipios.xlsx` carries no year at all. The old pipeline resolved it
  empirically: matched against an already-validated base, it agreed 100% on 499
  municipalities with **2020** (against 73% with 2019 and 67% with 2021). We carry
  that finding over rather than re-deriving it. Any *new* year-less file is skipped
  with a warning, because guessing a competence silently corrupts the series.
- **The Gini values are second-hand.** The original `ginibr.xlsx` no longer exists
  anywhere; these 645 values were recovered from the old project's derived
  parquet. Good enough to build the table, not good enough to cite. Same caveat as
  the IDH-M, which covers 642 of 645.
- **Gini and IDH-M are static**, keyed on `cod_ibge` alone — they are 2010 census
  values that never change. They are saved separately from the yearly CAPAG.

In [ ]:
# --- Socioeconomic: CAPAG (yearly) + IDH-M and Gini (static manual assets) ---
import re

# The one file the Treasury published with no year in its name. Resolved
# empirically against a validated base: 100% agreement on 499 municipalities with
# 2020, against 73% with 2019 and 67% with 2021. Carried over, not re-derived.
CAPAG_YEAR_BY_NAME = {"capag-municipios.xlsx": 2020}

def capag_year(filename: str) -> int | None:
    """Reference year from the file name, or None when it carries no year."""
    if filename in CAPAG_YEAR_BY_NAME:
        return CAPAG_YEAR_BY_NAME[filename]
    years = re.findall(r"20\d{2}", filename)
    return max(int(y) for y in years) if years else None

def read_capag(path, year: int) -> pd.DataFrame:
    """One CAPAG spreadsheet -> (cod_ibge, ano, capag) for SP.

    Header row and column names both move between editions, so we search for them
    instead of hard-coding. A file we cannot read returns empty rather than raising.
    """
    probe = pd.read_excel(path, header=None, engine="calamine", nrows=10)
    header_row = None
    for i in range(len(probe)):
        # str() per cell on purpose: pandas 3 keeps NA as NA through astype(str),
        # so a blank cell would come back as a float and break the search.
        cells = [str(v).lower() for v in probe.iloc[i].tolist()]
        if any("ibge" in c or "munic" in c for c in cells):
            header_row = i
            break
    if header_row is None:
        return pd.DataFrame()

    table = pd.read_excel(path, header=header_row, engine="calamine")
    code_col = next((c for c in table.columns
                     if any(k in str(c).lower() for k in ("ibge", "cód", "cod"))), None)
    grade_col = next((c for c in table.columns
                      if any(k in str(c).lower() for k in ("capag", "classif"))), None)
    if code_col is None or grade_col is None:
        return pd.DataFrame()

    out = table[[code_col, grade_col]].copy()
    out.columns = ["cod_ibge", "capag"]
    out = out.dropna(subset=["cod_ibge"])
    out["cod_ibge"] = (out["cod_ibge"].astype(str).str.strip()
                       .str.replace(r"\.0$", "", regex=True))
    out = out[out["cod_ibge"].str.startswith("35")]          # São Paulo
    out["cod_ibge"] = out["cod_ibge"].astype("int64")
    out["capag"] = out["capag"].astype(str).str.strip().str.upper()
    out = out[out["capag"].isin(["A", "B", "C", "D"])]       # drop notes and totals
    out["ano"] = year
    return out[["cod_ibge", "ano", "capag"]]

# Step 1: read every CAPAG spreadsheet.
capag_frames, skipped = [], []
for path in sorted((RAW_DIR / "socioeconomic" / "capag").glob("*.xls*")):
    year = capag_year(path.name)
    if year is None:
        skipped.append(path.name)
        continue
    frame = read_capag(path, year)
    print(f"  {year}  {path.name[:52]:<52} {len(frame):>4} SP municipalities")
    if not frame.empty:
        capag_frames.append(frame)

if skipped:
    print(f"\nSKIPPED (no year in the name, competence unknown): {skipped}")

capag = (pd.concat(capag_frames, ignore_index=True)
         .drop_duplicates(["cod_ibge", "ano"], keep="last")
         .sort_values(["cod_ibge", "ano"]).reset_index(drop=True))
print(f"\nCAPAG table: {capag.shape}  years {sorted(capag['ano'].unique())}")

# Step 2: the two static manual assets. Missing is EXPECTED here, not an error.
static_parts = []
for name, column in (("idhm_2010_sp.csv", "idhm_2010"), ("gini_2010_sp.csv", "gini_2010")):
    path = LOOKUP_DIR / name
    if path.exists():
        part = pd.read_csv(path).astype({"cod_ibge": "int64"})
        static_parts.append(part[["cod_ibge", column]])
        print(f"  {column}: {len(part)} of 645 municipalities  ({name})")
    else:
        print(f"  {column}: MISSING — expected, it is a manual asset ({path})")

socio_static = static_parts[0]
for part in static_parts[1:]:
    socio_static = socio_static.merge(part, on="cod_ibge", how="outer")
socio_static = socio_static.sort_values("cod_ibge").reset_index(drop=True)

# Step 3: look at what came out. CAPAG is yearly, the other two are static, so
# they are saved as two tables rather than one padded to a common key.
print(f"\nCAPAG grades by year:")
print(capag.pivot_table(index="ano", columns="capag", values="cod_ibge",
                        aggfunc="count", fill_value=0).to_string())
print(f"\nStatic table: {socio_static.shape}   columns: {list(socio_static.columns)}")

capag.to_parquet(OUT_DIR / "capag_wide.parquet", index=False)
socio_static.to_parquet(OUT_DIR / "socioeconomico_static.parquet", index=False)
print(f"Saved: capag_wide.parquet + socioeconomico_static.parquet")

socio_static.head(3)

### Water, sewage and drainage (SINISA)

Four indicators, all close to the mechanism of transmission. Unreliable piped
water is the direct cause of stored water — drums, tanks, cisterns — which is
where *Aedes* breeds; paving decides where standing water collects.

| Code | What it measures |
|---|---|
| `IAG0001` | population served by the water network (%) |
| `IES0001` | population served by the sewage collection network (%) |
| `IES0007` | households with sewage collected **and treated** (%) |
| `IAP0001` | paved public roads in the urban area (%) |

`01` already did the hard part — following the unpredictable links, unpacking the
`.zip`/`.rar` packages, and **finding the header row inside each spreadsheet**
(which moves between editions, and moves again when the Ministry publishes a
correction). It left `_indicators.csv`, one row per municipality per year per
indicator.

Things that matter here:

- **The file is national** (32,531 rows). We keep the São Paulo municipalities;
  the codes are already 7 digits.
- **Only 2023 and 2024 exist.** SINISA is a young system, so this source covers
  two years out of the eleven in the grid. Propagating it across the other years
  is Section 3's decision, not this cell's.
- **`IES0007`, not `IES2004`.** They are different concepts — coverage of the
  population versus efficiency of a treatment plant — and the old README claimed
  `IES2004` while the parser had always used `IES0007`. We keep `IES0007`, which is
  both what the models were trained on and the one closer to the mechanism.
- The long format becomes one column per indicator, so the key is `(cod_ibge, ano)`.

In [ ]:
# --- SINISA: water, sewage and urban drainage ---
sinisa_raw = pd.read_csv(RAW_DIR / "sinisa" / "_indicators.csv")
print(f"National rows: {len(sinisa_raw):,}   indicators: {sorted(sinisa_raw['indicador'].unique())}")
print(f"Years: {sorted(sinisa_raw['ano'].unique())}")

# São Paulo only; the codes are already 7 digits.
sinisa_sp = sinisa_raw[sinisa_raw["cod_ibge"].astype(str).str.startswith("35")].copy()
print(f"SP rows: {len(sinisa_sp):,}")

# Long -> wide: one column per indicator. `pivot_table` with "first" would hide a
# duplicate; `pivot` raises on one, which is what we want.
sinisa = (sinisa_sp
          .pivot(index=["cod_ibge", "ano"], columns="indicador", values="valor")
          .reset_index()
          .rename_axis(columns=None))
sinisa = sinisa.astype({"cod_ibge": "int64", "ano": "int64"})

# Keep the indicator codes in the names: they are the source's own vocabulary and
# the only unambiguous way to say which of the two sewage indicators this is.
sinisa = sinisa.rename(columns={
    "IAG0001": "iag0001_atend_agua_pct",
    "IES0001": "ies0001_atend_esgoto_pct",
    "IES0007": "ies0007_esgoto_tratado_pct",
    "IAP0001": "iap0001_vias_pavimentadas_pct",
}).sort_values(["cod_ibge", "ano"]).reset_index(drop=True)

print(f"\nSINISA table: {sinisa.shape}   columns: {list(sinisa.columns)}")
print("\nMunicipalities with a value, by year and indicator:")
print(sinisa.set_index(["cod_ibge", "ano"]).notna().groupby(level="ano").sum().to_string())

sinisa.to_parquet(OUT_DIR / "sinisa_wide.parquet", index=False)
print(f"Saved: {OUT_DIR / 'sinisa_wide.parquet'}")

sinisa.head(3)

### Land use and land cover (MapBiomas, Collection 11)

MapBiomas classifies every 30-metre pixel of Brazil from satellite imagery, year
by year since 1985, and publishes the areas aggregated per municipality. The link
to arboviruses runs both ways — deforestation pushes sylvatic yellow fever towards
people, urbanisation is what *Aedes aegypti* lives on — so it is one of the few
sources that speaks to dengue and yellow fever at once.

Things that matter here:

- **Collection 11 carries a `geocode` column — the 7-digit IBGE code.** The old
  pipeline matched municipalities **by name** against a lookup, which is fragile
  (accents, "Moji" vs "Mogi", homonyms across states). We join on the code and
  the whole class of error disappears.
- **The sheet is `COVERAGE_11`, not the first sheet.** The workbook opens on
  `READ_ME`; reading `sheet_name=0` gets you a page of prose.
- **Areas are in hectares, one row per (municipality, class), one column per
  year** (`y1985` … `y2025`). We convert to *percentage of the municipality's
  area*, which is what makes municipalities comparable and makes the unit
  irrelevant.
- **The percentage denominator is the sum of all classes**, including
  `7. Not Observed` (cloud, shadow). The cell prints how much that class accounts
  for; if it were large, the percentages would be quietly distorted.
- We keep only 2015 onward — the grid starts there, and melting 40 unused years
  would cost memory for nothing.

In [ ]:
# --- MapBiomas: land cover as a share of each municipality ---
mapbiomas_file = next((RAW_DIR / "mapbiomas").glob("*.xlsx"))
print(f"Reading: {mapbiomas_file.name}  (this is a ~78 MB workbook, it takes a moment)")

# The workbook opens on READ_ME; the data lives in the COVERAGE_* sheet.
from python_calamine import CalamineWorkbook
sheet = next(s for s in CalamineWorkbook.from_path(mapbiomas_file).sheet_names
             if s.upper().startswith("COVERAGE"))
print(f"Sheet: {sheet}")

YEARS = [y for y in range(2015, 2026)]
year_cols = [f"y{y}" for y in YEARS]
mb_raw = pd.read_excel(mapbiomas_file, sheet_name=sheet, engine="calamine",
                       usecols=["state", "geocode", "class_level_1"] + year_cols)

mb_sp = mb_raw[mb_raw["state"] == "São Paulo"].copy()
print(f"SP rows: {len(mb_sp):,}   classes: {sorted(mb_sp['class_level_1'].unique())}")

# Long format: one row per (municipality, class, year), area in hectares.
mb_long = mb_sp.melt(id_vars=["geocode", "class_level_1"], value_vars=year_cols,
                     var_name="ano", value_name="area_ha")
mb_long["ano"] = mb_long["ano"].str.lstrip("y").astype("int64")
mb_long["area_ha"] = mb_long["area_ha"].fillna(0)

# Sum the sub-classes into the six level-1 classes.
by_class = (mb_long.groupby(["geocode", "ano", "class_level_1"], as_index=False)["area_ha"]
            .sum())

# Percentage of the municipality's total area for that year. The denominator is
# every class, "Not Observed" included — see how big that is below.
total = by_class.groupby(["geocode", "ano"], as_index=False)["area_ha"].sum().rename(
    columns={"area_ha": "area_total_ha"})
by_class = by_class.merge(total, on=["geocode", "ano"], validate="m:1")
by_class["pct"] = (by_class["area_ha"] / by_class["area_total_ha"] * 100).round(4)

not_observed = by_class[by_class["class_level_1"].str.startswith("7.")]["pct"]
if not_observed.empty:
    print("\n'Not Observed' (cloud/shadow): not present in São Paulo at all — "
          "the percentages below are undistorted")
else:
    print(f"\n'Not Observed' share: mean {not_observed.mean():.4f}%, "
          f"max {not_observed.max():.4f}%  (the distortion it introduces)")

# Wide: one column per class, named the way the rest of the project names them.
CLASS_NAMES = {
    "1. Forest": "pct_floresta",
    "2. Herbaceous and Shrubby Vegetation": "pct_natural_nao_florestal",
    "3. Farming": "pct_agricultura",
    "4. Non vegetated area": "pct_nao_vegetado",
    "5. Water": "pct_agua",
}
by_class["coluna"] = by_class["class_level_1"].map(CLASS_NAMES)
mapbiomas = (by_class.dropna(subset=["coluna"])
             .pivot_table(index=["geocode", "ano"], columns="coluna", values="pct")
             .reset_index().rename_axis(columns=None)
             .rename(columns={"geocode": "cod_ibge"})
             .astype({"cod_ibge": "int64"})
             .sort_values(["cod_ibge", "ano"]).reset_index(drop=True))

print(f"\nMapBiomas table: {mapbiomas.shape}   columns: {list(mapbiomas.columns)}")
print(f"Municipalities: {mapbiomas['cod_ibge'].nunique()}  years: "
      f"{mapbiomas['ano'].min()}-{mapbiomas['ano'].max()}")
print("\nMean share by class (%):")
print(mapbiomas[list(CLASS_NAMES.values())].mean().round(2).to_string())

mapbiomas.to_parquet(OUT_DIR / "mapbiomas_wide.parquet", index=False)
print(f"Saved: {OUT_DIR / 'mapbiomas_wide.parquet'}")

mapbiomas.head(3)

### Municipal capacity and disaster history (IBGE MUNIC)

MUNIC is IBGE's survey of what municipal governments actually *have*: community
health agents, sanitary and epidemiological surveillance teams, endemic-disease
control staff, risk mapping, and a record of floods, landslides and droughts. It
is the closest thing we get to measuring whether a municipality can respond at all.

Things that matter here:

- **There is no series — each edition surveys different topics.** Health questions
  were asked in **2018 and 2021**; disaster questions only in **2020**. The 2019,
  2023 and 2024 editions contain none of our variables. `01` wrote
  `_editions.json` recording exactly which edition holds which variable and in
  which sheet, and this cell follows that map rather than assuming.
- **Column names change case between editions** — `MSAU28` in 2018, `Msau28` in
  2021 — and the municipality-code column is `Cod Municipio` in one and `CodMun`
  in another. Both are matched case-insensitively, by keyword.
- **"Não sabe", "Recusa" and "Não informou" become NaN, not False.** Only `Sim`
  and `Não` map to true/false. A municipality that refused to answer has *unknown*
  surveillance capacity, and recording that as "no capacity" would invent data.
  This is the same absence-vs-zero principle as everywhere else.
- **Each edition is a snapshot with a year**, so the key is `(cod_ibge, ano)` —
  but with only three usable years. Section 3 has to carry each snapshot forward
  to the next one; between 2018 and 2021, 165 of the 645 municipalities changed
  their answer on community health agents, so freezing one edition across the
  whole series would erase real change.

In [ ]:
# --- IBGE MUNIC: management, surveillance and disasters ---
import json

# Survey code -> the name we use downstream. The code stays in the name because
# it is the only unambiguous way back to the questionnaire.
MUNIC_COLUMNS = {
    "Msau28": "msau28_pacs",
    "Msau541": "msau541_vig_sanitaria",
    "Msau542": "msau542_vig_epidemiologica",
    "Msau543": "msau543_controle_endemias",
    "Mgrd01": "mgrd01_seca",
    "Mgrd06": "mgrd06_alagamento",
    "Mgrd07": "mgrd07_erosao",
    "Mgrd08": "mgrd08_enchente_gradual",
    "Mgrd11": "mgrd11_enxurrada",
    "Mgrd14": "mgrd14_deslizamento",
    "Mgrd201": "mgrd201_mapeamento_risco",
    "Mmam2612": "mmam2612_moradia_risco",
}
# Only a clear yes or no becomes a value. "Não sabe" / "Recusa" / "Não informou"
# fall through to NaN: unknown capacity is not the same as no capacity.
YES_NO = {"sim": True, "não": False, "nao": False}

munic_dir = RAW_DIR / "munic"
editions = json.loads((munic_dir / "_editions.json").read_text(encoding="utf-8"))
print(f"Editions collected: {sorted(editions)}")

frames = []
for year in sorted(editions):
    info = editions[year]
    if not info["variables"]:
        print(f"  {year}: none of our variables in this edition — skipped")
        continue

    # Group the variables by the sheet they live in, so each sheet is read once.
    by_sheet = {}
    for code, meta in info["variables"].items():
        by_sheet.setdefault(meta["sheet"], []).append(code)

    year_parts = []
    for sheet, codes in by_sheet.items():
        table = pd.read_excel(munic_dir / info["file"], sheet_name=sheet, engine="calamine")
        # Column names move between editions in case and spacing alike.
        lower = {str(c).strip().lower(): c for c in table.columns}
        code_col = next((lower[k] for k in lower
                         if k.startswith("cod") and "uf" not in k), None)
        if code_col is None:
            print(f"  {year}/{sheet}: no municipality-code column — skipped")
            continue

        picked = {}
        for code in codes:
            real = lower.get(code.lower())
            if real is None:
                print(f"  {year}/{sheet}: column {code} vanished — skipped")
                continue
            picked[real] = MUNIC_COLUMNS[code]

        part = table[[code_col] + list(picked)].rename(
            columns={code_col: "cod_ibge", **picked})
        part = part.dropna(subset=["cod_ibge"])
        part["cod_ibge"] = (part["cod_ibge"].astype(str).str.strip()
                            .str.replace(r"\.0$", "", regex=True))
        part = part[part["cod_ibge"].str.startswith("35")]         # São Paulo
        part["cod_ibge"] = part["cod_ibge"].astype("int64")
        for col in picked.values():
            part[col] = part[col].astype(str).str.strip().str.lower().map(YES_NO)
        year_parts.append(part.set_index("cod_ibge"))

    if not year_parts:
        continue
    merged = pd.concat(year_parts, axis=1).reset_index()
    merged["ano"] = int(year)
    frames.append(merged)
    answered = merged[[c for c in merged.columns if c not in ("cod_ibge", "ano")]]
    print(f"  {year}: {len(merged)} SP municipalities, "
          f"{answered.shape[1]} variable(s), "
          f"{answered.notna().to_numpy().mean():.1%} answered yes-or-no")

munic = (pd.concat(frames, ignore_index=True)
         .sort_values(["cod_ibge", "ano"]).reset_index(drop=True))
munic = munic[["cod_ibge", "ano"] + [c for c in MUNIC_COLUMNS.values() if c in munic.columns]]

print(f"\nMUNIC table: {munic.shape}   editions with data: {sorted(munic['ano'].unique())}")
print("\nNon-null values per variable, by edition:")
print(munic.set_index(["cod_ibge", "ano"]).notna().groupby(level="ano").sum().to_string())

munic.to_parquet(OUT_DIR / "munic_wide.parquet", index=False)
print(f"Saved: {OUT_DIR / 'munic_wide.parquet'}")

munic.head(3)

### Primary care coverage (e-Gestor APS)

The share of each municipality's population covered by the Family Health Strategy.
It matters twice over: primary care teams are who *detect* arbovirus cases, so
coverage shapes both the real burden and how much of it is ever reported.

`01` saved one JSON per month — 139 files, all 5,570 municipalities each.

Things that matter here:

- **There is a methodology break in 2021, and the two halves are not the same
  indicator.** Until 2020 the endpoint is `/ab` and the value is `pcCoberturaAb`;
  from 2021 it is `/aps` and the value is `qtCobertura`. Both are percentages —
  **despite the `qt` prefix, which normally means a count in this API**. We
  harmonise them into one column and keep `esf_metodologia` marking which regime
  each row came from, so the model sees the break instead of being fooled by it.
- **The AB half delivers every number as a US-formatted string** — comma for
  thousands, dot for decimals: `'56.88'`, `'1,134'`, `'11,895,893'`. The old
  pipeline's docstring called this "formato brasileiro", which is backwards, and
  acting on that reading turns a coverage of 56.88% into 5688. We strip the
  thousands commas and leave the dot alone.
- **AB is capped at 100%; APS is not.** Measured on this collection: the AB half
  never exceeds 100.00, while the APS half has a median of 108% and reaches 607%
  (Uru, 1,153 inhabitants, one team). APS reports *potential* coverage — a team's
  capacity divided by the population — so a single team in a tiny municipality
  can cover it several times over. This is a deeper break than the field rename:
  the two halves are not on the same scale, and a model that does not know about
  `esf_metodologia` will read the 2021 jump as an explosion of primary care.
  We keep the values uncapped, because capping would hide the break rather than
  handle it.
- **`qtEsf` (number of teams) exists in both regimes**, which makes it the one
  variable comparable straight across the break. That is worth more than it looks:
  it is the control that tells you whether a jump in coverage is real or an
  artefact of the change.
- **The file name carries the regime and the competence**, so we read the period
  from the name rather than from `nuComp` — whose format also changed, from
  `"201501"` to `"06/2026"`.
- Key is `(cod_ibge, ano, mes)`: this is one of the few genuinely monthly sources.

In [ ]:
# --- Primary care coverage (ESF/APS), 139 monthly JSON files ---
import json, re

esf_dir = RAW_DIR / "primary_care_coverage"
files = sorted(esf_dir.glob("cobertura_*.json"))
print(f"Monthly files: {len(files)}  ({files[0].name} ... {files[-1].name})")

# cobertura_ab_201501.json -> regime "ab", competence 2015-01
NAME = re.compile(r"cobertura_(ab|aps)_(\d{4})(\d{2})\.json$")

rows = []
for path in files:
    found = NAME.match(path.name)
    if not found:
        print(f"  unexpected file name, skipped: {path.name}")
        continue
    regime, year, month = found.group(1), int(found.group(2)), int(found.group(3))

    payload = json.loads(path.read_text(encoding="utf-8"))
    records = payload if isinstance(payload, list) else payload.get("dados", [])
    for rec in records:
        if str(rec.get("sgUf", "")).strip().upper() != "SP":
            continue
        # The value column is the first difference between the two regimes.
        coverage = rec.get("pcCoberturaAb") if regime == "ab" else rec.get("qtCobertura")
        rows.append({
            # coMunicipioIbge is the 6-DIGIT code, so it is named cod6 here.
            "cod6": rec.get("coMunicipioIbge"),
            "ano": year,
            "mes": month,
            "esf_cobertura_pct": coverage,
            "esf_qt_equipes": rec.get("qtEsf"),
            "esf_pop_referencia": rec.get("qtPopulacao"),
            "esf_metodologia": regime.upper(),
        })

esf = pd.DataFrame(rows)
print(f"SP rows collected: {len(esf):,}")

# The AB half delivers every number as a string in US format: comma for
# thousands, dot for decimals ('56.88', '1,134', '11,895,893'). The APS half
# delivers real numbers. Stripping the thousands commas handles both — and note
# this is NOT the Brazilian format the old pipeline's docstring claimed, which is
# how a percentage of 56.88 turns into 5688 if you "fix" the separators.
for column in ("esf_cobertura_pct", "esf_qt_equipes", "esf_pop_referencia"):
    esf[column] = pd.to_numeric(
        esf[column].astype(str).str.replace(",", "", regex=False), errors="coerce")
esf = esf.dropna(subset=["cod6"])
esf["cod6"] = esf["cod6"].astype(str).str.strip().str.zfill(6)
esf = esf.sort_values(["cod6", "ano", "mes"]).reset_index(drop=True)

# Step 2: look at the break. Comparing the two regimes side by side is the check
# that matters — if coverage jumps but the team count does not, the jump is an
# artefact of the methodology, not of reality.
print(f"\nESF table: {esf.shape}   columns: {list(esf.columns)}")
print(f"Municipalities: {esf['cod6'].nunique()}   months: {esf.groupby(['ano','mes']).ngroups}")
# Coverage is a percentage in both regimes, so a mean in the thousands would mean
# the parsing broke. Values ABOVE 100 are real in the APS half and expected —
# printed per regime so the difference in scale is impossible to miss.
print("\nCoverage (%) by regime — note AB stops at 100 and APS does not:")
print(esf.groupby("esf_metodologia")["esf_cobertura_pct"]
      .describe(percentiles=[.5, .9]).round(1).to_string())
print("\nBy regime:")
print(esf.groupby("esf_metodologia").agg(
    meses=("ano", lambda s: len(set(zip(s, esf.loc[s.index, "mes"])))),
    cobertura_media=("esf_cobertura_pct", "mean"),
    equipes_media=("esf_qt_equipes", "mean"),
).round(2).to_string())

esf.to_parquet(OUT_DIR / "esf_wide.parquet", index=False)
print(f"Saved: {OUT_DIR / 'esf_wide.parquet'}")

esf.head(3)

### Health network: hospital beds, and the real maternal mortality ratio

Three DATASUS datasets, and they do not share a cadence, so this becomes **two
tables**:

| Table | Key | What is in it |
|---|---|---|
| `leitos_wide` | (mun, ano, mes) | `leitos_sus` — SUS beds, from CNES/LT, monthly |
| `mortalidade_materna_wide` | (mun, ano) | `obitos_maternos`, `nascidos_vivos`, `rmm_por_100k_nv` |

**This is where the maternal mortality variable finally becomes what its name
says.** The old pipeline had a column called `mortalidade_materna` that contained
a raw *count* of deaths — the docstring promised a rate per 100,000 live births
and admitted, three lines later, that the denominator was never integrated. A
count of deaths mostly measures how many people live in the municipality. Now that
`01` collects SINASC, the denominator exists and the ratio is computed properly:

```
rmm_por_100k_nv = obitos_maternos / nascidos_vivos * 100000
```

**Both counts are kept as their own columns.** The ratio is a division of two
numbers that each carry meaning and error, and nobody should have to trust it
blind — with the numerator and denominator beside it, any surprising value can be
traced in one glance.

Things that matter here:

- **`DNSP2018.DBC` has an UPPERCASE extension.** Every other file is `.dbc`. A
  plain `glob("*.dbc")` silently loses the whole of 2018 — one year of the
  denominator, gone, with no error. We match case-insensitively.
- **`DTOBITO` and `DTNASC` are `DDMMYYYY`**, the opposite of `DT_NOTIFIC` (SINAN)
  and `DT_INTER` (SIH-SUS). We never parse them: both files are already one year
  each, so the year comes from the file name.
- **Numerator and denominator both end in 2024**, because SIM and SINASC each take
  about two years to consolidate. That alignment is the point — the ratio is never
  computed by dividing 2024 deaths by 2023 births. CNES beds, by contrast, run to
  2026: that is why they are a separate table rather than padded to match.
- **The ratio is unstable for small municipalities, and we do NOT fix that here.**
  Most municipalities have zero maternal deaths in a year; one death against 200
  births reads as 500 per 100,000, which is almost entirely noise. Whether to use
  a three-year window or a minimum denominator is a modelling decision and belongs
  to Section 5. The ingestion's job is to deliver the honest numbers and say so.
- Maternal deaths are chapter XV of ICD-10: `CAUSABAS` matching `O00`–`O99`.
- Beds are `QT_SUS` (SUS beds), not `QT_EXIST` (all beds), summed over every
  establishment and bed type in the municipality.

In [ ]:
# --- Health Step A: decode the .dbc files into per-file checkpoints ---
# Same pattern as SINAN and SIH: decode once, checkpoint, so a re-run of the
# aggregation below never pays the decoding cost again.
import gc, re, tempfile
from collections import Counter
from pathlib import Path

import dbfread
from pyreaddbc import dbc2dbf

health_ckpt = OUT_DIR / "health_checkpoints"
health_ckpt.mkdir(parents=True, exist_ok=True)

# Maternal deaths: ICD-10 chapter XV, O00-O99.
MATERNAL_ICD = re.compile(r"^O\d{2}")

# NOTE the case-insensitive suffix test: DNSP2018.DBC is uppercase while every
# other file is lowercase, and a plain glob("*.dbc") would drop 2018 in silence.
def dbc_files(folder: Path, prefix: str) -> list[Path]:
    return sorted(p for p in folder.iterdir()
                  if p.suffix.lower() == ".dbc" and p.name.upper().startswith(prefix))

beds_files = dbc_files(RAW_DIR / "health", "LTSP")
death_files = dbc_files(RAW_DIR / "health", "DOSP")
birth_files = dbc_files(RAW_DIR / "sinasc", "DNSP")
print(f"CNES/LT beds : {len(beds_files)} monthly files")
print(f"SIM deaths   : {len(death_files)} yearly files  {[p.stem[-4:] for p in death_files]}")
print(f"SINASC births: {len(birth_files)} yearly files  {[p.stem[-4:] for p in birth_files]}")

def read_dbc(path: Path):
    """Expand a .dbc into a temporary .dbf and yield its records (latin-1)."""
    with tempfile.TemporaryDirectory() as tmp:
        dbf = Path(tmp) / "x.dbf"
        dbc2dbf(str(path), str(dbf))
        for record in dbfread.DBF(str(dbf), encoding="latin1"):
            yield record

# --- Beds: sum QT_SUS per municipality, one file per month ---
for path in beds_files:
    found = re.match(r"LTSP(\d{2})(\d{2})", path.stem, re.I)
    year, month = 2000 + int(found.group(1)), int(found.group(2))
    out = health_ckpt / f"beds_{year}{month:02d}.parquet"
    if out.exists():
        continue
    totals = Counter()
    for record in read_dbc(path):
        code = str(record.get("CODUFMUN", "")).strip()
        beds = record.get("QT_SUS")
        if code and beds is not None:
            totals[code] += int(beds)
    pd.DataFrame({"cod6": list(totals), "leitos_sus": list(totals.values())}).assign(
        ano=year, mes=month).to_parquet(out, index=False)
    print(f"  beds {year}-{month:02d}: {len(totals)} municipalities", flush=True)
    gc.collect()

# --- Deaths and births: count per municipality of residence, one file per year ---
for kind, files, column in (("deaths", death_files, "obitos_maternos"),
                            ("births", birth_files, "nascidos_vivos")):
    for path in files:
        year = int(path.stem[-4:])
        out = health_ckpt / f"{kind}_{year}.parquet"
        if out.exists():
            continue
        totals = Counter()
        for record in read_dbc(path):
            # The year comes from the file name: DTOBITO and DTNASC are DDMMYYYY,
            # and reusing SINAN's YYYYMMDD parser here would read year 601.
            if kind == "deaths":
                cause = str(record.get("CAUSABAS", "")).strip().upper()
                if not MATERNAL_ICD.match(cause):
                    continue
            code = str(record.get("CODMUNRES", "")).strip().zfill(6)
            totals[code] += 1
        pd.DataFrame({"cod6": list(totals), column: list(totals.values())}).assign(
            ano=year).to_parquet(out, index=False)
        print(f"  {kind} {year}: {sum(totals.values()):,} records over "
              f"{len(totals)} municipalities", flush=True)
        gc.collect()

print(f"\nCheckpoints written to {health_ckpt}")

In [ ]:
# --- Health Step B: assemble the two tables ---

def load_checkpoints(prefix: str) -> pd.DataFrame:
    parts = [pd.read_parquet(p) for p in sorted(health_ckpt.glob(f"{prefix}_*.parquet"))]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

# --- Table 1: hospital beds, monthly ---
leitos = load_checkpoints("beds")
leitos = leitos[leitos["cod6"].str.startswith("35")].copy()          # São Paulo
leitos["cod6"] = leitos["cod6"].str.zfill(6)
leitos = (leitos.groupby(["cod6", "ano", "mes"], as_index=False)["leitos_sus"].sum()
          .sort_values(["cod6", "ano", "mes"]).reset_index(drop=True))

print(f"Beds table: {leitos.shape}")
print(f"  municipalities: {leitos['cod6'].nunique()}   months: "
      f"{leitos.groupby(['ano','mes']).ngroups}   "
      f"{leitos['ano'].min()}-{leitos['ano'].max()}")
leitos.to_parquet(OUT_DIR / "leitos_wide.parquet", index=False)

# --- Table 2: maternal mortality ratio, yearly ---
deaths = load_checkpoints("deaths")
births = load_checkpoints("births")
for frame in (deaths, births):
    frame["cod6"] = frame["cod6"].str.zfill(6)

# Outer join on purpose: a municipality-year with births but no maternal death is
# a real zero for the numerator, while one with deaths and no births would be a
# red flag worth seeing rather than dropping.
materna = births.merge(deaths, on=["cod6", "ano"], how="outer")
materna = materna[materna["cod6"].str.startswith("35")].copy()

# 350000 is DATASUS's "municipality unknown" — it belongs to no municipality.
unknown = int(materna.loc[materna["cod6"] == "350000",
                          ["nascidos_vivos", "obitos_maternos"]].sum().sum())
materna = materna[materna["cod6"] != "350000"]
print(f"\nDropped 'municipality unknown' (350000): {unknown} records")

# A municipality-year that appears in the births file with no maternal death row
# genuinely had zero maternal deaths — the file covers every birth, so absence of
# a death record is information, not a gap.
materna["obitos_maternos"] = materna["obitos_maternos"].fillna(0).astype("int64")
materna["rmm_por_100k_nv"] = (
    materna["obitos_maternos"] / materna["nascidos_vivos"] * 100_000).round(2)

materna = (materna[["cod6", "ano", "obitos_maternos", "nascidos_vivos",
                    "rmm_por_100k_nv"]]
           .sort_values(["cod6", "ano"]).reset_index(drop=True))

print(f"\nMaternal mortality table: {materna.shape}")
print(f"  years: {sorted(materna['ano'].unique())}")
print(f"  municipalities: {materna['cod6'].nunique()}")
print(f"  total births {materna['nascidos_vivos'].sum():,.0f} / "
      f"maternal deaths {materna['obitos_maternos'].sum():,}")

# The state-level ratio is the sanity check: São Paulo runs around 40-70 per
# 100,000, and any municipal number has to be read against that.
state = (materna.groupby("ano")[["obitos_maternos", "nascidos_vivos"]].sum())
state["rmm_estado"] = (state["obitos_maternos"] / state["nascidos_vivos"] * 100_000).round(1)
print("\nState-level ratio by year (the yardstick for the municipal numbers):")
print(state.to_string())

# And the instability, stated rather than smoothed away.
zero_deaths = (materna["obitos_maternos"] == 0).mean()
small = materna[materna["nascidos_vivos"] < 500]
print(f"\nMunicipality-years with zero maternal deaths: {zero_deaths:.1%}")
print(f"Municipality-years with fewer than 500 births: {len(small)} "
      f"({len(small)/len(materna):.1%}) — in these, one death moves the ratio by "
      f"{100_000/500:.0f}+ points. Smoothing is Section 5's call, not this cell's.")

materna.to_parquet(OUT_DIR / "mortalidade_materna_wide.parquet", index=False)
print(f"\nSaved: leitos_wide.parquet + mortalidade_materna_wide.parquet")

materna.head(3)

### Yellow fever vaccination coverage — stitching two systems

The last and most awkward source: no single system holds the series, because the
Ministry of Health moved the whole thing mid-way. **This stitching does not exist
in the old pipeline** — it read a single hand-made CSV. We write it from scratch.

| | TabNet PNI (1994–2022) | InfoMS/RNDS panel (2023–2026) |
|---|---|---|
| Granularity | **yearly** | **monthly, by cohort** |
| Coverage | percentage, comma decimal (`"316,27"`) | ratio (`0.9833`) — needs ×100 |
| Municipality | inside the text (`"110001 ALTA FLORESTA D'OESTE"`) | its own column |
| Extras | none | doses applied, cohort population |

**How the two are made comparable.** The panel's monthly "População" is not the
municipal population — it moves month to month (11,111 → 10,453 → 12,134 …)
because it is the **target cohort for that month**. So the monthly coverages
cannot be averaged, and dividing a year's doses by one month's population gives
absurdities (954% for the capital in 2023). The right annual figure is

```
coverage = sum(doses over the year) / sum(cohort population over the year) * 100
```

which is exactly the formula TabNet itself uses — the query captured from its form
is `sum(QT_DOSE)*100/sum(NU_POPULACAO)`. The two halves are therefore built the
same way, from different systems.

Things that matter here:

- **There is no overlapping year, so the size of the break cannot be measured.**
  TabNet ends in 2022 and the panel starts in 2023. With a shared year we could
  have calibrated classic PNI against RNDS; without one, we can only mark where
  the seam is. The `fonte` column carries it, and any analysis crossing 2022→2023
  has to treat that as a discontinuity rather than a trend.
- **Values above 100% are real and are kept uncapped.** The target denominator is
  an estimate that lags reality, so a municipality that vaccinates migrants or
  neighbours can exceed its own target. Capping would hide a genuine property of
  the data. What we do instead is *print the distribution*, so an implausible
  spread is visible before anything is saved.
- **TabNet uses the comma as the decimal separator** — the opposite of the e-Gestor
  files, which are US-formatted. Getting this backwards turns 316,27% into 31,627%.
- **The full TabNet series from 1994 is kept**, even though the grid starts in
  2015. It costs nothing and the earlier years are real data; Section 3 trims.

In [ ]:
# --- Yellow fever vaccination: stitch TabNet (yearly) + Qlik panel (monthly) ---
import re

vac_dir = RAW_DIR / "yellow_fever_vaccination"

# ---------- Half 1: TabNet, 1994-2022, already yearly ----------
tabnet_file = next(vac_dir.glob("cob_vac_fa_tabnet_*.csv"))
tabnet_raw = pd.read_csv(tabnet_file, sep=";", encoding="latin1", dtype=str)
print(f"TabNet: {tabnet_file.name}  {tabnet_raw.shape}")

# The municipality column holds "110001 ALTA FLORESTA D'OESTE": a 6-digit code
# followed by the name. The last row is a national "Total" and has no code.
tabnet_raw["cod6"] = tabnet_raw["Município"].str.extract(r"^(\d{6})")
tabnet_raw = tabnet_raw.dropna(subset=["cod6"])

year_cols = [c for c in tabnet_raw.columns if re.fullmatch(r"\d{4}", c.strip())]
tabnet = tabnet_raw.melt(id_vars="cod6", value_vars=year_cols,
                         var_name="ano", value_name="cob_vac_fa_pct")
tabnet["ano"] = tabnet["ano"].str.strip().astype("int64")

# TabNet writes the decimal with a COMMA — the opposite of the e-Gestor files.
tabnet["cob_vac_fa_pct"] = pd.to_numeric(
    tabnet["cob_vac_fa_pct"].str.strip().str.replace(",", ".", regex=False),
    errors="coerce")
tabnet["vacinacao_fonte"] = "tabnet"   # not "fonte": IBGE ships one too
tabnet = tabnet[tabnet["cod6"].str.startswith("35")]
print(f"  SP rows: {len(tabnet):,}  years {tabnet['ano'].min()}-{tabnet['ano'].max()}")

# ---------- Half 2: the Qlik panel, monthly by cohort -> yearly ----------
panel_parts = []
for path in sorted(vac_dir.glob("cob_vac_painel_*.xlsx")):
    year = int(re.search(r"(\d{4})", path.stem).group(1))
    sheet = pd.read_excel(path, engine="calamine", header=None)

    # Three header rows: vaccine, month, measure. Data starts on the fourth.
    measures = [str(v) for v in sheet.iloc[2].tolist()]
    codes = sheet.iloc[3:, 1].astype(str).str.extract(r"^(\d{6})")[0]
    body = sheet.iloc[3:].reset_index(drop=True)

    dose_cols = [j for j, m in enumerate(measures) if m == "Doses Aplicadas"]
    pop_cols = [j for j, m in enumerate(measures) if m == "População"]

    doses = body.iloc[:, dose_cols].apply(pd.to_numeric, errors="coerce").sum(axis=1)
    cohort = body.iloc[:, pop_cols].apply(pd.to_numeric, errors="coerce").sum(axis=1)

    part = pd.DataFrame({"cod6": codes.values, "ano": year,
                         "doses": doses.values, "cohort": cohort.values})
    part = part.dropna(subset=["cod6"])          # drops the totals row (no code)
    print(f"  panel {year}: {len(dose_cols)} months, {len(part)} municipalities")
    panel_parts.append(part)

panel = pd.concat(panel_parts, ignore_index=True)
panel = panel[panel["cod6"].str.startswith("35")]

# The annual figure: doses over the year divided by the summed monthly cohorts.
# Averaging the monthly coverages would be wrong (each month has its own, moving
# denominator), and dividing by one month's cohort gives nonsense.
# A cohort of zero makes the ratio undefined, not infinite: a municipality with
# doses recorded against no target population tells us nothing about coverage.
# Guard it, or the value comes out as inf and poisons every later statistic.
panel["cob_vac_fa_pct"] = (
    panel["doses"] / panel["cohort"].where(panel["cohort"] > 0) * 100).round(2)
sem_coorte = int((panel["cohort"] == 0).sum())
if sem_coorte:
    print(f"  {sem_coorte} municipality-year(s) with a ZERO cohort -> coverage left undefined")
panel["vacinacao_fonte"] = "painel"
print(f"  SP rows: {len(panel):,}  years {panel['ano'].min()}-{panel['ano'].max()}")

# ---------- Stack ----------
vacinacao = (pd.concat([tabnet[["cod6", "ano", "cob_vac_fa_pct", "vacinacao_fonte"]],
                        panel[["cod6", "ano", "cob_vac_fa_pct", "vacinacao_fonte"]]],
                       ignore_index=True)
             .sort_values(["cod6", "ano"]).reset_index(drop=True))

# ---------- The check that decides whether this is right ----------
# Vaccination coverage should sit roughly between 60% and 120%; above 100% is
# legitimate because the target denominator lags. A median in the hundreds would
# mean the formula is still wrong, and nothing should be saved on that.
print(f"\nVaccination table: {vacinacao.shape}")
print("\n=== Coverage distribution per municipality-year, by source ===")
print(vacinacao.groupby("vacinacao_fonte")["cob_vac_fa_pct"]
      .describe(percentiles=[.05, .5, .95]).round(1).to_string())

print("\n=== By year (median, and the tails) ===")
by_year = vacinacao.groupby(["vacinacao_fonte", "ano"])["cob_vac_fa_pct"].agg(
    n="count", minimo="min", mediana="median", maximo="max").round(1)
print(by_year.loc[by_year.index.get_level_values("ano") >= 2014].to_string())

extreme = (vacinacao["cob_vac_fa_pct"] > 300).sum()
print(f"\nMunicipality-years above 300%: {extreme} "
      f"({extreme / len(vacinacao):.2%}) — a handful is the lagging denominator; "
      f"a systematic pattern would mean the formula is still wrong")

vacinacao.to_parquet(OUT_DIR / "vacinacao_fa_wide.parquet", index=False)
print(f"\nSaved: {OUT_DIR / 'vacinacao_fa_wide.parquet'}")

vacinacao.head(3)

### Commuting between municipalities (IBGE Census 2022, SIDRA)

How many residents leave their own municipality to work somewhere else. It belongs
here because arboviruses travel with people: a municipality that sends thousands
of commuters into a neighbour every morning is epidemiologically closer to it than
the map suggests.

**Scope decision — read this before using the variable.** This enters the table as
a **single 2022 snapshot of the OUTBOUND flow only**, static per municipality. Two
things that the old pipeline had are deliberately not here:

- **No historical series.** The old base reconstructed 2015–2021 from the 2010
  Census *sample microdata* — 2 GB of fixed-width text that has to be parsed
  against a layout dictionary. We measured what that buys first: in the old
  project's own ablation, removing both commuting variables moves dengue AUROC in
  the **fourth decimal place** (−0.0004 to +0.0005 across the four outbreak
  definitions), and the whole `mobilidade` group ranks 6th of 9. The one large
  number in that table is a degenerate target where *removing* the variable
  improved the model, which is variance, not signal.
- **No inbound flow.** `pendulares_entram_trabalho` requires the origin-destination
  matrix, which only the microdata carries. The 2022 microdata were still not
  public as of this collection, so reprocessing 2010 would have reproduced exactly
  the years the old base already had, without extending anything forward.

So the cost was real and the gain was measured to be nil at this stage. **The
historical series and the inbound flow are future work**, and if they are ever
needed the raw microdata are already downloaded in `raw/commuting/`.

Things that matter about what *is* here:

- The SIDRA variable counts employed residents aged 10+ whose main job is in
  **another municipality**, by municipality of **residence** — so it is genuinely
  an outbound flow, not a workplace count.
- `localidade_id` is already the 7-digit code.
- Non-numeric SIDRA codes (`-`, `..`, `...`, `X`) become null, never zero — the
  same rule as every other SIDRA source in this notebook.
- Being static, it joins on `cod_ibge` alone and repeats across every row of a
  municipality. Section 3 should treat it as a structural constant, and any
  analysis that reads it as time-varying is reading something that is not there.

In [ ]:
# --- Commuting: outbound flow, 2022 Census snapshot (SIDRA table 10329) ---
import json

commuting_file = RAW_DIR / "commuting" / "sidra_10329_saidas_2022.json"
payload = json.loads(commuting_file.read_text(encoding="utf-8"))
print(f"Reading: {commuting_file.name}")
print(f"Variable: {payload[0]['variavel'][:90]}...")
print(f"Unit: {payload[0].get('unidade')}")

# SIDRA never leaves a cell blank; it sends a code saying why it is blank.
NON_NUMERIC = {"-", "..", "...", "X", "x", ""}

rows = []
for block in payload[0]["resultados"]:
    for item in block.get("series", []):
        raw_value = (item.get("serie") or {}).get("2022")
        rows.append({
            "cod_ibge": int(item["localidade"]["id"]),
            "pendulares_saem_trabalho":
                None if str(raw_value).strip() in NON_NUMERIC else raw_value,
        })

mobilidade = pd.DataFrame(rows)
mobilidade["pendulares_saem_trabalho"] = pd.to_numeric(
    mobilidade["pendulares_saem_trabalho"], errors="coerce")
mobilidade = mobilidade.sort_values("cod_ibge").reset_index(drop=True)

# Look at what came out. Static table: one row per municipality, no time key.
print(f"\nCommuting table: {mobilidade.shape}   columns: {list(mobilidade.columns)}")
print(f"Municipalities: {len(mobilidade)}  (expected 645)")
print(f"With a value: {mobilidade['pendulares_saem_trabalho'].notna().sum()}")
print(f"Outbound commuters: min {mobilidade['pendulares_saem_trabalho'].min():,.0f}, "
      f"median {mobilidade['pendulares_saem_trabalho'].median():,.0f}, "
      f"max {mobilidade['pendulares_saem_trabalho'].max():,.0f}")

# The largest flows should be the dormitory towns of Greater São Paulo — if the
# top of this list looks like anywhere else, the variable is not what we think.
top = mobilidade.nlargest(5, "pendulares_saem_trabalho")
print("\nLargest outbound flows (expect Greater São Paulo dormitory cities):")
print(top.to_string(index=False))

mobilidade.to_parquet(OUT_DIR / "mobilidade_wide.parquet", index=False)
print(f"\nSaved: {OUT_DIR / 'mobilidade_wide.parquet'}")

mobilidade.head(3)

### NASA POWER — climate (temperature, rain, humidity, wind, pressure)

Climate is a first-line predictor for arboviruses: Aedes transmission depends
directly on temperature and rainfall. Monthly series per municipality, 2015-01
onward, already keyed on the 7-digit `cod_ibge` (no 6→7 needed) and with full
coverage (every municipality-month has climate). This source was collected but not
yet ingested — it comes already clean, so ingestion is just read + confirm + save.

In [22]:
# --- NASA POWER: already clean, keyed on cod_ibge, monthly. Just load and save. ---
nasa_power_wide = pd.read_parquet(RAW_DIR / "nasa_power" / "nasa_power_municipios_sp.parquet")

# sanity checks
print(f"Rows: {len(nasa_power_wide):,}  (grid is 89,010; extra rows are months beyond the cutoff)")
print(f"Municipalities: {nasa_power_wide['cod_ibge'].nunique()}  (expected 645)")
print(f"Year range: {nasa_power_wide['ano'].min()}–{nasa_power_wide['ano'].max()}")
print(f"Temp range (t2m): {nasa_power_wide['t2m'].min():.1f} to {nasa_power_wide['t2m'].max():.1f} °C")
print(f"Any nulls in t2m? {nasa_power_wide['t2m'].isna().sum()}")

nasa_power_wide.to_parquet(OUT_DIR / "nasa_power_wide.parquet", index=False)
print(f"\nSaved: nasa_power_wide.parquet")

Rows: 89,655  (grid is 89,010; extra rows are months beyond the cutoff)
Municipalities: 645  (expected 645)
Year range: 2015–2026
Temp range (t2m): 12.3 to 31.8 °C
Any nulls in t2m? 0

Saved: nasa_power_wide.parquet


## Section 3 — The join: 18 tables become one

Left join of each source onto the grid — **18 parquets in 17 joins** (SINAN goes
in first, in the cell below; socioeconomic and health each produce two tables,
since their halves have different cadences). The order actually executed:

1. **Monthly**, on `(cod_ibge, ano, mes)`: NASA POWER → SIH-SUS → beds → ESF →
   yellow fever.
2. **Yearly**, on `(cod_ibge, ano)`: IBGE → **density** → CAPAG → SINISA →
   MapBiomas → MUNIC.
3. **Housing**, expanded to the grid years *before* its join, because one of its
   two census years (2010) falls outside the grid.
4. **Yearly, after the 6→7 conversion**: maternal mortality → vaccination.
5. **Forward-fill** of the per-edition snapshots (MUNIC, SINISA).
6. **Static**, on `cod_ibge` alone: gini and IDH-M → commuting → territorial areas.

Note that **density is yearly, not static**: population moves every year, so each
year carries its own value rather than one frozen number repeated back to 2015.

**Locked corrections:**
- **`validate='m:1'` on EVERY join** — in the old pipeline 7 of the 15 sources
  lacked this guard, and its absence once nearly multiplied rows (the CAPAG source
  with two revisions of the same year, 397 duplicates). The guard makes the join
  shout if a duplicate breaks cardinality, instead of silently inflating the table.
- **Gini as static by `cod_ibge`** (alongside density), not via the IBGE yearly
  join — otherwise a 2010 census value, immutable by definition, vanished in the
  2026 months through a plumbing accident.
- Yearly → monthly: the yearly value repeats across the 12 months (merge on year).
- Per-edition source: each snapshot holds until the next (ffill + bfill), because
  freezing on the first edition would erase real change between censuses.

In [24]:
import pandas as pd

OUT_DIR = sinan_ckpt.parent   # data/master/

# --- Build the 6-digit -> 7-digit map from the INMET lookup ---
cod6_to_cod7 = {str(int(c))[:-1]: int(c) for c in municipios["cod_ibge"]}
print(f"6->7 map has {len(cod6_to_cod7)} municipalities (expected 645)")

def cod6_to_cod7_col(df):
    """Add a 7-digit cod_ibge from cod6; drop rows with no match."""
    df = df.copy()
    df["cod_ibge"] = df["cod6"].astype(str).str.strip().map(cod6_to_cod7)
    n_before = len(df)
    df = df.dropna(subset=["cod_ibge"])
    df["cod_ibge"] = df["cod_ibge"].astype(int)
    return df.drop(columns=["cod6"]), n_before - len(df)

# --- Test the conversion on SINAN ---
sinan_wide = pd.read_parquet(OUT_DIR / "sinan_wide.parquet")
sinan_7, dropped = cod6_to_cod7_col(sinan_wide)
print(f"SINAN: {len(sinan_wide)} rows in, {dropped} dropped (no match), {len(sinan_7)} kept")
sinan_7.head(3)

# --- First real join: SINAN onto the grid, with cardinality guard ---
# The grid is one row per (cod_ibge, ano, mes). SINAN is also one row per that key
# (after 6->7). So it's many-grid-rows-to-one-sinan-row: validate='m:1'.

master = grid.merge(
    sinan_7,
    on=["cod_ibge", "ano", "mes"],
    how="left",
    validate="m:1",     # <- guards against duplicates silently multiplying rows
)

print(f"Grid rows before join: {len(grid):,}")
print(f"Master rows after SINAN join: {len(master):,}")
print(f"Same? {len(grid) == len(master)}  <- MUST be True")
print(f"\nMunicipality-months with dengue cases: {master['dengue_casos'].notna().sum():,}")
print(f"(the rest are NaN — no notification, handled in Section 4)")
master[["cod_ibge", "municipio", "ano", "mes", "dengue_casos"]].head(3)

6->7 map has 645 municipalities (expected 645)
SINAN: 48831 rows in, 33 dropped (no match), 48798 kept
Grid rows before join: 89,010
Master rows after SINAN join: 89,010
Same? True  <- MUST be True

Municipality-months with dengue cases: 47,411
(the rest are NaN — no notification, handled in Section 4)


,cod_ibge,municipio,ano,mes,dengue_casos
0,3500105,Adamantina,2015,1,9.0
1,3500105,Adamantina,2015,2,48.0
2,3500105,Adamantina,2015,3,347.0


In [ ]:
# --- Section 3, monthly sources: key (cod_ibge, ano, mes) ---
# Every join is a left join onto `master`, and every one carries validate="m:1".
# The row count is printed after each: it must never move off the grid size.
EXPECTED_ROWS = len(grid)
print(f"Grid size that every join must preserve: {EXPECTED_ROWS:,}\n")

def join_source(master, name, keys, needs_cod6=False):
    """Left-join one parquet onto master and prove the row count did not move."""
    table = pd.read_parquet(OUT_DIR / f"{name}.parquet")
    dropped = 0
    if needs_cod6:
        table, dropped = cod6_to_cod7_col(table)
    new_cols = [c for c in table.columns if c not in keys]

    before = len(master)
    master = master.merge(table, on=keys, how="left", validate="m:1")

    status = "OK " if len(master) == EXPECTED_ROWS else "!!!"
    print(f"[{status}] {name:<26} +{len(new_cols):>2} cols  "
          f"rows {before:,} -> {len(master):,}"
          + (f"  ({dropped} rows dropped with no 6->7 match)" if dropped else ""))
    if len(master) != EXPECTED_ROWS:
        raise RuntimeError(
            f"{name} changed the row count: {before:,} -> {len(master):,}. "
            f"That is a duplicate key slipping past the guard — stop and inspect.")
    return master

MONTH_KEY = ["cod_ibge", "ano", "mes"]
master = join_source(master, "nasa_power_wide", MONTH_KEY)
master = join_source(master, "sih_wide", MONTH_KEY, needs_cod6=True)
master = join_source(master, "leitos_wide", MONTH_KEY, needs_cod6=True)
master = join_source(master, "esf_wide", MONTH_KEY, needs_cod6=True)
master = join_source(master, "yellow_fever_wide", MONTH_KEY, needs_cod6=True)

print(f"\nAfter the monthly sources: {master.shape}")

In [ ]:
# --- Section 3, yearly sources: key (cod_ibge, ano) ---
# The yearly value repeats across the 12 months of its year, which is what the
# merge on (cod_ibge, ano) does on its own. Years the source never published
# simply stay NaN — no filling here, that is Section 4.
YEAR_KEY = ["cod_ibge", "ano"]

for name in ("ibge_wide", "densidade_wide", "capag_wide", "sinisa_wide",
             "mapbiomas_wide", "munic_wide"):
    master = join_source(master, name, YEAR_KEY)

# --- Housing needs expanding BEFORE the join, not filling after it ---
# Its two census years are 2010 and 2022, and 2010 is OUTSIDE the grid. Joining
# on year would drop the 2010 snapshot entirely, and a later bfill would then
# project 2022 backwards over 2015-2021 — losing the 8 municipalities that have
# a 2010 record and no 2022 one. So we reindex each municipality over the census
# years AND the grid years first, and let each snapshot hold until the next:
# 2010 covers 2015-2021, 2022 covers 2022 onward. This is what the old pipeline
# did in `_juntar_por_edicao`, and keeping it makes the base comparable.
habitacao_raw = pd.read_parquet(OUT_DIR / "habitacao_wide.parquet")
grid_years = sorted(master["ano"].unique())
HOUSING_COLS = ["num_favelas", "pop_favelas"]

all_years = sorted(set(habitacao_raw["ano"]) | set(grid_years))
full_index = pd.MultiIndex.from_product(
    [sorted(habitacao_raw["cod_ibge"].unique()), all_years], names=["cod_ibge", "ano"])
habitacao_expanded = (habitacao_raw.set_index(["cod_ibge", "ano"])[HOUSING_COLS]
                      .reindex(full_index)
                      .groupby(level="cod_ibge").ffill()
                      .groupby(level="cod_ibge").bfill()
                      .reset_index())
habitacao_expanded = habitacao_expanded[habitacao_expanded["ano"].isin(grid_years)]

com_registro = habitacao_raw["cod_ibge"].nunique()
so_2010 = (set(habitacao_raw.loc[habitacao_raw["ano"] == 2010, "cod_ibge"])
           - set(habitacao_raw.loc[habitacao_raw["ano"] == 2022, "cod_ibge"]))
print(f"\nHousing expanded: {com_registro} municipalities with any census record "
      f"({len(so_2010)} of them only in 2010, recovered by this step)")

before_rows = len(master)
master = master.merge(habitacao_expanded, on=YEAR_KEY, how="left", validate="m:1")
status = "OK " if len(master) == EXPECTED_ROWS else "!!!"
print(f"[{status}] {'habitacao (expanded)':<26} + 2 cols  "
      f"rows {before_rows:,} -> {len(master):,}")
if len(master) != EXPECTED_ROWS:
    raise RuntimeError("housing join changed the row count")

master = join_source(master, "mortalidade_materna_wide", YEAR_KEY, needs_cod6=True)
master = join_source(master, "vacinacao_fa_wide", YEAR_KEY, needs_cod6=True)

print(f"\nAfter the yearly sources: {master.shape}")

# How much of the grid each yearly source actually reaches. A source published for
# only a few years covers only those years — that is a fact about the source, and
# whether to carry values forward is Section 4's decision, not this cell's.
print("\nGrid coverage of the yearly sources (share of the 89,010 rows):")
for col in ("pop_residente", "densidade_hab_km2", "capag", "iag0001_atend_agua_pct",
            "pct_floresta", "msau28_pacs", "num_favelas", "rmm_por_100k_nv",
            "cob_vac_fa_pct"):
    if col in master.columns:
        filled = master[col].notna().sum()
        years = sorted(master.loc[master[col].notna(), "ano"].unique())
        span = f"{years[0]}-{years[-1]}" if years else "none"
        print(f"  {col:<26} {filled:>6,} rows ({filled/len(master):5.1%})  years {span}")

### Carrying the per-edition sources forward

Three sources are **snapshots, not series**: they exist only for the years their
survey ran. Joined on `(cod_ibge, ano)` they land on those years and leave the
rest of the grid empty — housing reaches 1.2% of the rows, MUNIC health 17.4%,
sanitation 16.9%. That is not missing data in the ordinary sense: a municipality's
favela count in 2019 is not unknown, it is *the 2018 figure, still standing until
the next census*.

So each snapshot is carried forward until the next one, and back-filled before the
first, per municipality. **Housing is handled separately, in the join cell above**,
because one of its two census years (2010) falls outside the grid: expanding it
after the join would have silently dropped that snapshot and the 8 municipalities
that appear only in it.

| Source | Published for | Columns carried |
|---|---|---|
| MUNIC (IBGE survey) | 2018, 2020, 2021 | `msau*`, `mgrd*`, `mmam*` (12) |
| SINISA (sanitation) | 2023, 2024 | `iag0001*`, `ies0001*`, `ies0007*`, `iap0001*` |

The back-fill matters as much as the forward-fill: without it the whole of
2015–2017 would be empty for MUNIC and 2015–2022 for SINISA. Assuming structural
stability over a few years is the most conservative option available — the
alternative is discarding those years entirely.

**What deliberately does NOT get filled, and why:**

- **Cases, deaths and hospitalisations** (SINAN, SIH-SUS, yellow fever). A NaN
  there means *no notification in that municipality-month*, which is information
  about the epidemic, not a gap in a structural variable. Carrying last month's
  case count forward would fabricate an outbreak. Section 4 decides what those
  NaNs become.
- **Population, GDP, CAPAG, density, vaccination, MapBiomas.** Each of these
  already has its own value for each year it was published; a gap in them is a
  year the source genuinely did not publish, and filling it here would hide that.
- **Beds and primary-care coverage**, which are monthly and real for every month.

In [ ]:
# --- Carry the per-edition (snapshot) sources across the grid ---
# Only these three sources. Everything else keeps its NaN — see the markdown above.
# Housing is NOT here: it was expanded before its join (see the cell above),
# because its 2010 snapshot falls outside the grid and would be lost otherwise.
SNAPSHOT_COLUMNS = {
    "munic": [c for c in master.columns
              if c.startswith(("msau", "mgrd", "mmam"))],
    "sinisa": [c for c in master.columns
               if c.startswith(("iag0001", "ies0001", "ies0007", "iap0001"))],
}
to_fill = [c for cols in SNAPSHOT_COLUMNS.values() for c in cols]
print(f"Carrying forward {len(to_fill)} columns from 3 snapshot sources:")
for source, cols in SNAPSHOT_COLUMNS.items():
    print(f"  {source:<10} {len(cols):>2} columns")

before = {c: master[c].notna().mean() for c in to_fill}

# Sort first: ffill walks rows in order, so the frame has to be in time order
# within each municipality or it would carry values backwards through time.
master = master.sort_values(["cod_ibge", "ano", "mes"]).reset_index(drop=True)

# ffill carries each snapshot until the next one; bfill then reaches the years
# before the first snapshot. Grouped by municipality so nothing ever leaks from
# one municipality into another.
grouped = master.groupby("cod_ibge", sort=False)[to_fill]
master[to_fill] = grouped.ffill()
grouped = master.groupby("cod_ibge", sort=False)[to_fill]
master[to_fill] = grouped.bfill()

print(f"\nRows after the fill: {len(master):,}  (must still be {EXPECTED_ROWS:,}: "
      f"{len(master) == EXPECTED_ROWS})")

print("\n=== Coverage before -> after ===")
for source, cols in SNAPSHOT_COLUMNS.items():
    for col in cols:
        print(f"  {col:<30} {before[col]:6.1%} -> {master[col].notna().mean():6.1%}")

# The three the review asked for. num_favelas comes from the expansion above,
# not from this fill, and lands at ~15.5%: the municipalities that actually have
# a census record. The rest have NO record, which stays NaN and never becomes 0.
print("\nThe three checked by hand:")
for col in ("num_favelas", "msau28_pacs", "iag0001_atend_agua_pct"):
    print(f"  {col:<30} {master[col].notna().mean():6.1%}")

In [ ]:
# --- Section 3, static sources: key (cod_ibge) only ---
# One value per municipality, repeated down every one of its months. Note the key
# is cod_ibge ALONE — passing the full triple here would match nothing.
STATIC_KEY = ["cod_ibge"]
for name in ("socioeconomico_static", "mobilidade_wide", "territorial_areas_wide"):
    master = join_source(master, name, STATIC_KEY)

# --- The final shape ---
print(f"\n{'='*70}")
print(f"MASTER: {master.shape[0]:,} rows x {master.shape[1]} columns")
print(f"Rows match the grid exactly: {len(master) == EXPECTED_ROWS}")
print(f"Municipalities: {master['cod_ibge'].nunique()}   "
      f"months: {master.groupby(['ano','mes']).ngroups}   "
      f"span: {master['ano'].min()}-{master['ano'].max()}")
print("=" * 70)

print(f"\nAll {len(master.columns)} columns:")
for i, col in enumerate(master.columns, 1):
    filled = master[col].notna().mean()
    print(f"  {i:>3}. {col:<34} {filled:6.1%} filled")

master.head(3)

### Population: filling the one year IBGE never published

Population is the denominator of incidence, so it feeds straight into the
`inc100` and `inc300` outbreak definitions in Section 5. A municipality-month
without it produces a NaN label, not a negative one.

**The hole is 2023, and only 2023.** The estimates series (SIDRA 6579) skips both
2022 and 2023; the 2022 Census *count* (table 4709) fills 2022, which leaves 2023
alone with nothing published. Every other year of the grid — 2015 through 2022 and
2024 through 2026 — has a real published figure.

This is worth stating because the old pipeline's comment said the opposite: it
described the gap as "2024-2025" and forward-filled to cover it. That was true of
the data available when it was written; on this collection, 2024, 2025 and 2026
are all published and 2023 is the one missing.

So the forward-fill carries the **2022 census count into 2023**, per municipality.
It is the conservative choice: municipal population typically moves less than 2%
between consecutive years, well inside the error of the estimate it replaces.
`pop_fonte` is deliberately left empty on those rows — a filled population with no
source recorded is exactly how you spot a carried-forward value later.

Note what is **not** filled: `pib_mil_reais` ends in 2023 and stays empty for
2024-2026. There the numerator is missing rather than the denominator, it does not
touch the labels, and inventing GDP would be a different and worse decision.

In [ ]:
# --- Population forward-fill, then save the joined master ---
from pathlib import Path

# Sorted by time within each municipality, so ffill moves forward in time only.
master = master.sort_values(["cod_ibge", "ano", "mes"]).reset_index(drop=True)

missing_before = sorted(master.loc[master["pop_residente"].isna(), "ano"].unique())
print(f"Years with no published population: {missing_before}")

master["pop_residente"] = master.groupby("cod_ibge", sort=False)["pop_residente"].ffill()

missing_after = sorted(master.loc[master["pop_residente"].isna(), "ano"].unique())
print(f"After the fill: {missing_after if missing_after else 'no year left empty'}")
print(f"pop_residente filled: {master['pop_residente'].notna().mean():.1%}")

# The rows that were filled carry a population but no pop_fonte — that empty
# source IS the marker saying the value was carried forward, not published.
carried = master["pop_residente"].notna() & master["pop_fonte"].isna()
print(f"Rows carrying a forward-filled population: {carried.sum():,} "
      f"({sorted(master.loc[carried, 'ano'].unique())})")

# --- Final shape, then save ---
print(f"\n{'='*70}")
print(f"MASTER (post-join, pre-labels): {master.shape[0]:,} rows x {master.shape[1]} columns")
print(f"Rows match the grid: {len(master) == EXPECTED_ROWS}")
print("=" * 70)

master_path = OUT_DIR / "municipio_mes_raw.parquet"
master.to_parquet(master_path, index=False)

print(f"\nSaved to: {master_path}")
print(f"Exists on disk: {Path(master_path).exists()}")
print(f"Size: {Path(master_path).stat().st_size / 1e6:.1f} MB")

# Read it back — the only proof that what is on disk is what we built.
check = pd.read_parquet(master_path)
print(f"Read back: {check.shape}  identical shape: {check.shape == master.shape}")

master.head(3)

## Section 4 — Missing vs zero (a documented decision, not a mass conversion)

We do **not** convert NaN to zero in the master. The rule:
- **Event columns** (cases, deaths, hospitalizations): a NaN means "not notified".
  Inside the frozen grid (already trimmed by the freshness cutoff), a non-notified
  month is a **true zero**, so these become 0 **locally** when computing outbreak
  labels (Section 5) and features (03) — never mass-converted here.
- **Structural columns** (favelas, SUS beds, risk mapping): a NaN means "unknown"
  (no census record, no SUS hospital, or "don't know" on a survey). These **stay
  NaN** — converting to zero would falsely assert "none".

This cell audits that both kinds of NaN coexist correctly, and confirms the
population denominator is complete (needed for incidence in Section 5).

In [30]:
# --- Section 4: absence vs zero — audit, not conversion ---
# Decision: we do NOT mass-convert NaN to zero in the master. NaN stays NaN here.
# Conversion to zero happens LOCALLY, only where it's a true zero:
#  - outbreak labels (Section 5): cases.fillna(0), because inside the frozen grid a
#    non-notified month IS a true zero.
#  - features (03_train): same logic.
# Structural NaN (no census record, municipality without SUS beds) is NEVER zero —
# it means "unknown", not "none".

# Audit: show where NaN lives, to confirm the two kinds coexist correctly.
def pct_filled(col):
    return 100 * master[col].notna().mean()

print("Event columns (NaN = not notified — will become 0 in labels):")
for c in ["dengue_casos", "zika_casos", "chikungunya_casos", "fa_casos"]:
    print(f"  {c:20} {pct_filled(c):5.1f}% filled")

print("\nStructural columns (NaN = unknown — stays NaN):")
for c in ["num_favelas", "leitos_sus", "mmam2612_moradia_risco"]:
    print(f"  {c:24} {pct_filled(c):5.1f}% filled")

print("\nDenominator (must be 100% for incidence to compute):")
print(f"  pop_residente        {pct_filled('pop_residente'):5.1f}% filled")

Event columns (NaN = not notified — will become 0 in labels):
  dengue_casos          53.3% filled
  zika_casos             5.4% filled
  chikungunya_casos      4.1% filled
  fa_casos               0.3% filled

Structural columns (NaN = unknown — stays NaN):
  num_favelas               15.5% filled
  leitos_sus                55.2% filled
  mmam2612_moradia_risco    61.2% filled

Denominator (must be 100% for incidence to compute):
  pop_residente        100.0% filled


## Section 5 — Outbreak labels

For each disease, we compute 4 outbreak definitions, all also requiring **≥ 5
cases** (so a small municipality with 1 isolated case isn't flagged). Labels go to
a separate `labels.parquet` (the targets), keeping the master as the features.

The 4 definitions:
- **inc100**: incidence ≥ 100 per 100k inhabitants
- **inc300**: incidence ≥ 300 per 100k
- **canal** (endemic channel): cases > historical median + 1.96·σ
- **zscore**: (cases − mean)/σ > 2

`canal` and `zscore` use a **baseline** (median, mean, σ per municipality-month)
built **without the epidemic years** — otherwise the epidemic itself would
contaminate the notion of "normal". Labels are computed for all years; only the
baseline excludes the epidemic ones.

We start with **inc100** (simplest, and it validates the population denominator).
Here a NaN in cases becomes 0 — inside the frozen grid, a non-notified month is a
true zero (the Section 4 decision, applied locally).

In [31]:
# --- Section 5: outbreak labels. Start the labels table with the key. ---
labels = master[["cod_ibge", "ano", "mes"]].copy()

MIN_CASES = 5   # a municipality-month needs at least 5 cases to be an outbreak

def incidence(disease):
    """Incidence per 100k. NaN cases -> 0 (true zero inside the grid)."""
    cases = master[f"{disease}_casos"].fillna(0)
    return cases / master["pop_residente"] * 100_000

# --- Definition inc100, for dengue first ---
disease = "dengue"
cases = master[f"{disease}_casos"].fillna(0)
inc = incidence(disease)

labels[f"{disease}_inc100"] = ((inc >= 100) & (cases >= MIN_CASES)).astype(int)

# sanity check
n_outbreak = labels[f"{disease}_inc100"].sum()
print(f"{disease} inc100 outbreaks: {n_outbreak:,} of {len(labels):,} municipality-months "
      f"({100*n_outbreak/len(labels):.1f}%)")
print(f"Outbreaks by year:")
print(labels.groupby(master['ano'])[f'{disease}_inc100'].sum())

dengue inc100 outbreaks: 14,218 of 89,010 municipality-months (16.0%)
Outbreaks by year:
ano
2015    1768
2016     550
2017       2
2018      93
2019    1586
2020    1037
2021     702
2022    1575
2023    1289
2024    3019
2025    2398
2026     199
Name: dengue_inc100, dtype: int64


In [34]:
# --- Endemic channel: cases > baseline median + 1.96·sigma ---
# The baseline (per municipality, per calendar month) is built WITHOUT epidemic
# years, so the epidemic itself doesn't inflate what counts as "normal".

EPIDEMIC_YEARS = {
    "dengue": [2015, 2019, 2024, 2025],
    "chikungunya": [2021, 2024, 2025],
    "zika": [2016],
    "fa": [2017, 2018, 2019, 2025],   # yellow-fever columns use the "fa_" prefix
}

disease = "dengue"
cases_col = f"{disease}_casos"

# frame with cases (NaN -> 0, true zero inside the grid) + calendar month
tmp = master[["cod_ibge", "ano", "mes"]].copy()
tmp["cases"] = master[cases_col].fillna(0)

# 1) baseline: exclude epidemic years, then per (municipality, calendar month)
#    compute median and sigma of the non-epidemic history
non_epidemic = tmp[~tmp["ano"].isin(EPIDEMIC_YEARS[disease])]
baseline = (
    non_epidemic.groupby(["cod_ibge", "mes"])["cases"]
    .agg(base_median="median", base_std="std")
    .reset_index()
)
# std is NaN when a (mun, month) has only one non-epidemic observation -> treat as 0
baseline["base_std"] = baseline["base_std"].fillna(0)

# 2) bring the baseline back to every row (labels ARE computed for epidemic years too;
#    only the baseline excluded them)
tmp = tmp.merge(baseline, on=["cod_ibge", "mes"], how="left", validate="m:1")

# 3) the endemic-channel threshold and the label
threshold = tmp["base_median"] + 1.96 * tmp["base_std"]
labels[f"{disease}_canal"] = ((tmp["cases"] > threshold) & (tmp["cases"] >= MIN_CASES)).astype(int)

# sanity check
n = labels[f"{disease}_canal"].sum()
print(f"{disease} canal outbreaks: {n:,} ({100*n/len(labels):.1f}% of full base)")
print(f"Prevalence over test years 2022-2024: "
      f"{100*labels.loc[master['ano'].isin([2022,2023,2024]), f'{disease}_canal'].mean():.1f}%")
print("Outbreaks by year:")
print(labels.groupby(master['ano'])[f'{disease}_canal'].sum())

dengue canal outbreaks: 12,881 (14.5% of full base)
Prevalence over test years 2022-2024: 26.5%
Outbreaks by year:
ano
2015    1410
2016     309
2017      29
2018      91
2019    1297
2020     586
2021     478
2022    1509
2023    1221
2024    3428
2025    2407
2026     116
Name: dengue_canal, dtype: int64


### All four definitions, for all four diseases

The two cells above worked one definition at a time and validated dengue against
the reference (inc100 25.3%, canal 26.5% over the test years). The cell below
generalises the same arithmetic to the full 4 × 4 grid, and **recomputes the two
validated dengue columns as a self-check**: if the generalisation changed them,
something in it is wrong and the assertion says so before anything is saved.

Two things carry over unchanged from the validated versions:

- **The baseline excludes the epidemic years; the labels do not.** A baseline
  built over an epidemic would call the epidemic normal. But every year gets a
  label, epidemic years included — they are exactly the periods the model has to
  learn to recognise.
- **σ = 0 has an explicit fallback.** When a municipality-month has a flat
  non-epidemic history, the z-score is undefined (division by zero). The
  documented rule is: any month above that constant mean counts as the anomaly,
  still subject to the ≥ 5 cases floor.

**What to expect from the rare diseases.** Yellow fever has 733 cases in the whole
base and zika 18,458, against dengue's 5.4 million. For those, `inc100` and
`inc300` will be zero or near zero almost everywhere — an incidence of 100 per
100,000 needs roughly 100 cases in a municipality of 100,000, which simply does
not happen for yellow fever. That is not a bug: it is the definitions disagreeing
about what an outbreak of a rare disease is, which is precisely what the
definition-sensitivity question is about.

In [ ]:
# --- All 4 definitions x 4 diseases ---
import numpy as np

DISEASES = ["dengue", "zika", "chikungunya", "fa"]
DEFINITIONS = ["inc100", "inc300", "canal", "zscore"]

# Snapshot the two already-validated dengue columns, to prove the generalisation
# reproduces them exactly rather than quietly drifting.
validated = {c: labels[c].copy() for c in ("dengue_inc100", "dengue_canal")
             if c in labels.columns}

def baseline_stats(disease):
    """Median, mean and sigma per (municipality, calendar month), computed WITHOUT
    that disease's epidemic years, then broadcast back to every row of the grid."""
    frame = master[["cod_ibge", "ano", "mes"]].copy()
    frame["cases"] = master[f"{disease}_casos"].fillna(0)

    non_epidemic = frame[~frame["ano"].isin(EPIDEMIC_YEARS[disease])]
    stats = (non_epidemic.groupby(["cod_ibge", "mes"])["cases"]
             .agg(base_median="median", base_mean="mean", base_std="std")
             .reset_index())
    # sigma is NaN when a (municipality, month) has a single non-epidemic
    # observation. Treated as 0, which routes it to the flat-baseline fallback.
    stats["base_std"] = stats["base_std"].fillna(0)

    return frame.merge(stats, on=["cod_ibge", "mes"], how="left", validate="m:1")

for disease in DISEASES:
    cases = master[f"{disease}_casos"].fillna(0)
    incid = cases / master["pop_residente"] * 100_000
    enough = (cases >= MIN_CASES).to_numpy()          # the floor, shared by all four

    # --- Raw incidence thresholds ---
    labels[f"{disease}_inc100"] = ((incid.to_numpy() >= 100) & enough).astype(int)
    labels[f"{disease}_inc300"] = ((incid.to_numpy() >= 300) & enough).astype(int)

    # --- Baseline-relative definitions ---
    base = baseline_stats(disease)
    observed = base["cases"].to_numpy()
    median = base["base_median"].to_numpy()
    mean = base["base_mean"].to_numpy()
    sigma = base["base_std"].to_numpy()

    labels[f"{disease}_canal"] = (
        (observed > median + 1.96 * sigma) & enough).astype(int)

    # Where sigma > 0 the z-score is defined; where it is 0 the baseline is flat
    # and any month above the mean is the anomaly.
    with np.errstate(divide="ignore", invalid="ignore"):
        z_exceeds = np.where(sigma > 0,
                             (observed - mean) / np.where(sigma > 0, sigma, np.nan) > 2,
                             observed > mean)
    labels[f"{disease}_zscore"] = (z_exceeds & enough).astype(int)

    print(f"{disease:<12} done  ({int(cases.sum()):,} cases in the base)")

# --- Self-check against the validated columns ---
for col, before in validated.items():
    identical = bool((labels[col] == before).all())
    print(f"\nReproduces the validated {col}: {identical}")
    if not identical:
        raise RuntimeError(
            f"{col} changed when generalised — the 4x4 loop does not match the "
            f"cell that was validated against the reference. Stop and compare.")

print(f"\nLabels table: {labels.shape}   ({len(DISEASES)} diseases x "
      f"{len(DEFINITIONS)} definitions = {len(DISEASES)*len(DEFINITIONS)} label columns)")

In [ ]:
# --- Prevalence over the test years, 2022-2024 ---
TEST_YEARS = [2022, 2023, 2024]
in_test = master["ano"].isin(TEST_YEARS).to_numpy()
print(f"Test window: {TEST_YEARS}  ({in_test.sum():,} municipality-months)\n")

rows = []
for disease in DISEASES:
    row = {"doenca": disease}
    for definition in DEFINITIONS:
        share = labels.loc[in_test, f"{disease}_{definition}"].mean() * 100
        row[definition] = round(share, 2)
    rows.append(row)

prevalence = pd.DataFrame(rows).set_index("doenca")
print("Prevalence (% of municipality-months flagged as an outbreak):")
print(prevalence.to_string())

# Call out the empty cells rather than leaving a silent 0.00 to be misread.
empty = [(d, s) for d in DISEASES for s in DEFINITIONS
         if prevalence.loc[d, s] == 0]
if empty:
    print("\nDefinitions that flag NOTHING in the test window:")
    for disease, definition in empty:
        total = int(master[f"{disease}_casos"].fillna(0)[in_test].sum())
        if definition.startswith("inc"):
            why = (f"an absolute incidence of {definition[3:]}/100k is never reached "
                   f"by a disease with {total:,} cases across the whole window")
        else:
            why = (f"with {total:,} cases in the window, no municipality-month clears "
                   f"the {MIN_CASES}-case floor on top of its baseline")
        print(f"  {disease}_{definition:<7} — {why}. Expected, not a failure.")

In [ ]:
# --- Cohen's kappa between the four definitions, for dengue ---
# The question: how much do the definitions agree BEYOND chance? Two definitions
# that both flag ~25% of months will agree ~60% of the time by luck alone, so raw
# agreement flatters them. Kappa removes that.
from sklearn.metrics import cohen_kappa_score
from itertools import combinations

disease = "dengue"
print(f"Cohen's kappa between definitions — {disease}, test years {TEST_YEARS}")
print("  <0.20 poor · 0.21-0.40 fair · 0.41-0.60 moderate · "
      "0.61-0.80 substantial · 0.81+ almost perfect\n")

kappa_matrix = pd.DataFrame(index=DEFINITIONS, columns=DEFINITIONS, dtype=float)
for a, b in combinations(DEFINITIONS, 2):
    ya = labels.loc[in_test, f"{disease}_{a}"]
    yb = labels.loc[in_test, f"{disease}_{b}"]
    kappa = cohen_kappa_score(ya, yb)
    kappa_matrix.loc[a, b] = kappa_matrix.loc[b, a] = round(kappa, 3)
    print(f"  {a:<7} vs {b:<7}  kappa = {kappa:+.3f}   "
          f"(flag {ya.mean()*100:.1f}% and {yb.mean()*100:.1f}%)")
# Not np.fill_diagonal(kappa_matrix.values, ...): pandas 3 hands back a
# read-only array, so the diagonal is set through the accessor instead.
for definition in DEFINITIONS:
    kappa_matrix.loc[definition, definition] = 1.0

print("\nMatrix:")
print(kappa_matrix.to_string())

In [ ]:
# --- Save the labels ---
from pathlib import Path

labels_path = OUT_DIR / "labels.parquet"
labels = labels.sort_values(["cod_ibge", "ano", "mes"]).reset_index(drop=True)
labels.to_parquet(labels_path, index=False)

print(f"Saved to: {labels_path}")
print(f"Exists on disk: {Path(labels_path).exists()}")
print(f"Shape: {labels.shape[0]:,} rows x {labels.shape[1]} columns")
print(f"Columns: {list(labels.columns)}")

# Read back — the only proof that what is on disk is what we built.
check = pd.read_parquet(labels_path)
print(f"\nRead back: {check.shape}  identical: {check.equals(labels)}")
print(f"Key unique: {not check.duplicated(['cod_ibge','ano','mes']).any()}")

labels.head(3)

## Section 6 — Internal consistency checks

The pipeline verifies itself — no dependency on any external or frozen reference.
These checks run every time and guard the master's integrity:
- the key (cod_ibge, ano, mes) is unique;
- the grid is complete (645 × 138 = 89,010 rows);
- the population denominator has no gaps (incidence always computable);
- every outbreak label is 0 or 1;
- no municipality-month is flagged as an outbreak with fewer than 5 cases
  (the minimum-cases rule held).

(During development the re-collection was also diffed against an earlier frozen
snapshot of this base, to confirm nothing had drifted unexpectedly. Every
difference was explainable: identical in the consolidated years, small revisions
in 2015-2019 where DATASUS reclassified SINAN records, and fuller 2026 coverage
because the re-collection is more recent. That was a one-off check, not part of
the pipeline.)

In [40]:
# --- Section 6: internal consistency checks (self-contained, permanent) ---
checks = []

# 1) unique key
dup = master.duplicated(subset=["cod_ibge", "ano", "mes"]).sum()
checks.append(("unique key (cod_ibge, ano, mes)", dup == 0, f"{dup} duplicates"))

# 2) grid complete
checks.append(("grid = 89,010 rows", len(master) == 89_010, f"{len(master)} rows"))
checks.append(("645 municipalities", master["cod_ibge"].nunique() == 645, f"{master['cod_ibge'].nunique()}"))

# 3) population complete
pop_gaps = master["pop_residente"].isna().sum()
checks.append(("population has no gaps", pop_gaps == 0, f"{pop_gaps} gaps"))

# 4) labels are binary
label_cols = [c for c in labels.columns if c not in ("cod_ibge", "ano", "mes")]
all_binary = all(set(labels[c].dropna().unique()) <= {0, 1} for c in label_cols)
checks.append(("all labels in {0,1}", all_binary, f"{len(label_cols)} label cols"))

# 5) min-cases rule held: no outbreak with <5 cases
violations = 0
for disease in ["dengue", "zika", "chikungunya", "fa"]:
    cases = master[f"{disease}_casos"].fillna(0)
    for defn in ["inc100", "inc300", "canal", "zscore"]:
        col = f"{disease}_{defn}"
        if col in labels.columns:
            violations += ((labels[col] == 1) & (cases < 5)).sum()
checks.append(("no outbreak with <5 cases", violations == 0, f"{violations} violations"))

# report
print("Internal consistency checks:")
all_ok = True
for name, ok, detail in checks:
    status = "PASS" if ok else "FAIL"
    print(f"  [{status}] {name}  ({detail})")
    all_ok = all_ok and ok
print(f"\n{'ALL CHECKS PASSED' if all_ok else 'SOME CHECKS FAILED — investigate'}")

Internal consistency checks:
  [PASS] unique key (cod_ibge, ano, mes)  (0 duplicates)
  [PASS] grid = 89,010 rows  (89010 rows)
  [PASS] 645 municipalities  (645)
  [PASS] population has no gaps  (0 gaps)
  [PASS] all labels in {0,1}  (16 label cols)
  [PASS] no outbreak with <5 cases  (0 violations)

ALL CHECKS PASSED


## Section 7 — Output

Two final files, the standard ML split of features (X) and targets (y):
- `municipio_mes.parquet` — the master: one row per municipality-month, all
  feature variables (climate, cases, hospitalizations, socioeconomic, etc.).
- `labels.parquet` — the 16 outbreak labels (4 diseases × 4 definitions).

They're kept separate on purpose: it makes it physically impossible for a label to
leak into training as if it were a feature. `03_train` loads X once and picks the
target(s) to model.

In [41]:
# --- Section 7: finalize outputs ---
# The master (features) already exists as municipio_mes_raw.parquet from Section 3.
# Rename it to the final name (drop "raw" — it's the finished master now).
import shutil

raw_path = OUT_DIR / "municipio_mes_raw.parquet"
final_path = OUT_DIR / "municipio_mes.parquet"
shutil.copy(raw_path, final_path)   # copy (keep raw as an intermediate audit trail)

# Confirm both final files
X = pd.read_parquet(final_path)
y = pd.read_parquet(OUT_DIR / "labels.parquet")

print("=== Master ready for 03_train ===")
print(f"X (features): {X.shape}  ->  {final_path.name}")
print(f"y (labels):   {y.shape}  ->  labels.parquet")
print(f"\nKeys match: {(X[['cod_ibge','ano','mes']].equals(y[['cod_ibge','ano','mes']]))}")
print(f"Feature columns: {X.shape[1] - 4} (excluding cod_ibge, municipio, ano, mes)")
print(f"Label columns: {y.shape[1] - 3} (excluding cod_ibge, ano, mes)")

=== Master ready for 03_train ===
X (features): (89010, 83)  ->  municipio_mes.parquet
y (labels):   (89010, 19)  ->  labels.parquet

Keys match: True
Feature columns: 79 (excluding cod_ibge, municipio, ano, mes)
Label columns: 16 (excluding cod_ibge, ano, mes)
